# 🎬 画像と音声から「話す人物動画（長尺対応）」を作る — かんたん操作ガイド（Context Loop対応）

**MiniMax H3 ＋ LatentSync 1.6｜Google Colab A100向け｜v13.1 Motion Context長尺対応（再開処理修正）（日本語操作ガイド付き）**

**どんな人物・キャラクターでも共通で使えます。** 入力画像と音声を差し替えて、新しい動画を作成できます。出力ファイル名や保存先も特定のキャラクター名に依存しません。

**初めての方は、この説明 → ①アップロード → ②設定 → 各セルを実行、の順でOKです。** 難しい英語の設定名を覚える必要はありません。①・②の入力欄には、何のための設定か日本語で説明しています。

> ✅ **v10で実際に動画生成成功。** 添付された2026年10月25日の実行レポートでは、A100で15秒のH3動画を1本作り、同じ映像から口パク設定 **1.5 / 1.8 / 2.0** の完成MP4を3本生成。**全て15秒・音声との尺の差0秒**でした。口の見た目や日本語の発話への追従精度までは、この数値だけでは判定できません。
>
> **v13では、15秒を超えるH3新規生成にMotion Contextによる連続生成を追加しました。** H3／LatentSyncの推論アルゴリズム、パラメータの初期値、分割・結合の処理はv11から変更していません。v13でのA100実行は未確認です。

---

## 🆕 長い音声の扱い（v13）

- **例：40秒の音声** → クリップ数・各クリップの秒数を自動決定 → H3で続き生成 → 重複映像を切り落として結合 → LatentSync → **40秒の元音声で完成**。
- 長尺生成の初期設定は **`H3_CONTINUITY = 自動（15秒超で使用）`**。15秒以下の処理は従来どおりです。
- Motion Contextでは **映像22フレーム（約0.92秒）** を前のクリップから引き継ぎます。2本目以降の生成尺にはこの重複分も含めるため、実際の分割尺は15秒より短くなることがあります。
- **中断したら同じ⑥を再実行**：生成済みMP4＋潜在データが残っていれば次のクリップから再開。**動画だけ残って潜在データがない場合は安全のため、そのクリップから再生成**します。
- **既存のv12ジョブは自動的に連続生成へ変わりません**。新しい長尺モードで作るにはH3映像を新規生成してください。既存動画からのLatentSync再実行は従来どおり利用できます。

## まず、何ができる？

**①画像＋日本語音声を渡す → ②体や表情を動かす → ③声に合わせて口元を修正 → ④MP4を保存**

```text
人物画像 ─┐
         ├─ MiniMax H3（表情・頭・手・体の動きを作る）
日本語音声┘          ↓
              LatentSync（元音声に合わせて口元を直す）
                      ↓
              元の日本語音声を付けた完成MP4
```

- **声は元の日本語音声を使います。** H3が作った音声を最終音声として採用しません。
- **15秒を超える素材は、前クリップの動きをMotion Contextで引き継ぎながら自動生成して結合**します。境目の不自然さを抑える目的ですが、完全な継ぎ目なしを保証するものではありません。
- **H3の映像は使い回せます。** 口パクの設定だけ変えるときに、H3の重い映像生成をやり直す必要はありません。

## 迷ったら、下の3つから選んでください

| やりたいこと | ①アップロード | ②で選ぶ「実行モード」 | 実行するセル |
|:--|:--|:--|:--|
| **A. 初めて動画を作る** | 画像＋元音声 | **H3生成から実行**（初期値） | **① → ② → ③ → ④ → ⑤ → ⑥ → ⑦ → ⑧ → ⑨ → ⑩-A → ⑩-B → ⑪ → ⑫** |
| **B. さっき作ったH3を使って口パクだけ再実行** | 同じ元音声（①の「前回のアップロードを使う」も可） | **保存済みH3ジョブから再開** | **① → ② → ⑦-B → ⑧* → ⑨* → ⑩-A → ⑩-B → ⑪ → ⑫** |
| **C. 手元の動画に口パクを付け直す** | 元音声＋動画MP4（画像不要） | **既存H3動画から再開** | **① → ② → ⑦-B → ⑧ → ⑨ → ⑩-A → ⑩-B → ⑪ → ⑫** |

**＊⑧・⑨は、そのColabランタイムでLatentSyncの準備が済んでいれば通常は再実行不要です。** ただし、ランタイムを再起動したときは環境を作り直してください。同じランタイムで口パク設定だけ変えた場合は、まず②を変更して、⑪ → ⑫を再実行するだけで済むケースもあります。

⚠️ **「保存済みH3ジョブ」はColabの `/content` にあるファイルを使う方式です。** ランタイム切断後も必ず残るわけではありません。別のランタイムで再利用したいときは、H3のMP4を手元に保存し、**C**の方法で使ってください。また、Bでは前回の音声と今回の音声が同じか確認し、異なる場合は安全のため停止します。

## はじめて動かすときの5つのポイント

1. Colabの **［ランタイム］→［ランタイムのタイプを変更］** で **A100 GPU** を選ぶ（利用可能な場合）。
2. **①** で画像と音声をアップロード。画像と音声は別々に選べます。
3. **②** は、初回ならそのままでOK。試運転は **TEST_SECONDS=5～15** 秒、全尺は **0** 秒です。
4. 各セルの左の **▶再生ボタン** を、上の表の順に押します。画面の **✅ 成功** や **⏭ スキップ** を確認してください。
5. 最後の **⑫** に、完成MP4・レポート・ZIPへのリンクが出ます。必要なファイルを選んで保存してください。

## 設定はここだけ覚えればOK

| 設定（②にあります） | わかりやすい意味 | 迷ったら |
|:--|:--|:--|
| `PIPELINE_MODE` | 新しくH3動画を作るか、既存の動画を使うか | 最初は「H3生成から実行」 |
| `TEST_SECONDS` | 先頭から何秒を作るか | お試し **15**／全部 **0** |
| `H3_CONTINUITY` | 15秒超の映像を前の動きを引き継ぎながら作るか | **自動（ON）** |
| `H3_CONTEXT_FRAMES` | 次のクリップへ渡す映像の重複フレーム数 | **22** |
| `STEP_MODE` | **体・表情を作るH3** の処理量 | **8ステップ** |
| `LIPSYNC_STEPS` | **口元を修正するLatentSync** の処理量 | **20** |
| `LIPSYNC_GUIDANCE` | 口元生成を音声でどの程度誘導するか | **1.5**（見た目で比較） |
| `COMPARE_LIPSYNC` | 口パクの設定違いをまとめて作るか | 通常 **OFF**／比較 **ON** |
| `COMPARE_GUIDANCES` | 比較したい誘導係数をカンマで指定 | **1.5,1.8,2.0** |
| `LIPSYNC_DEEPCACHE` | 口パク処理の高速化を試す | **ON** |

**通常は比較OFFで完成MP4が1本、比較ONなら設定の数だけ作成します。** 例えば、口パク20ステップ・誘導係数3種類なら3本分のGPU推論が必要です。H3は共通の1本を再利用します。

## どのセルが何をする？

| セル | 役割 | 時間がかかる処理？ |
|:--|:--|:--|
| **①** | 画像・音声・必要ならMP4をアップロード | 通信速度次第 |
| **②** | モードや長さ、品質を設定 | ほぼかからない |
| **③～⑤** | H3を動かす環境・モデルの準備 | 初回はダウンロードあり |
| **⑥** | H3で表情・手などが動く映像を作る | **GPU処理・重い** |
| **⑦** | H3映像の仮完成版を確認 | 動画変換 |
| **⑦-B** | 以前作ったH3／手元のMP4を読み込む | ⑥の再生成はしない |
| **⑧～⑨** | LatentSyncの環境・モデルの準備 | 初回はダウンロードあり |
| **⑩-A・⑩-B** | 動画と音声の整列、顔を検出できるか確認 | 前処理・テスト |
| **⑪** | 口元を生成・設定違いを比較 | **GPU処理・重い** |
| **⑫** | 完成MP4の結合、比較MP4・レポート・ZIPを作る | 動画変換 |

## 実測時間の目安（15秒・A100 40GB）

今回の成功レポートでは、H3 **約230秒**、LatentSync **約154秒 × 3設定**、合計**約693秒（11分33秒）のモデル処理**でした。**セットアップ・モデルダウンロード・動画の最終変換などは含みません。** 口パク設定を1種類だけにすれば、その分の比較推論は不要です。ColabのGPU環境や混雑状況で所要時間・CU消費は変わります。

## 保存先・出力ファイルの見かた

- **作業用フォルダは `/content/h3_latentsync_workspace/`**。入力素材は `inputs/`、生成結果は `outputs/` に保存します。各実行でジョブ別のフォルダができるため、素材ごとに区別できます。
- **以前の版の保存済みジョブ**を使う場合は、②の「保存済みH3ジョブの場所」にそのジョブのフォルダをフルパスで指定してください。新しい作業用フォルダからは以前の保存先を自動検索しません。

- **`H3_LatentSync_..._Final.mp4`**：完成版。ファイル名の `s20`＝口パク20ステップ、`g1p8`＝誘導係数1.8、`dc1`＝DeepCache ON。
- **`MiniMax_H3_with_original_audio.mp4`**：H3映像に元の日本語音声を載せた口パク修正前の比較用動画。
- **`comparison_side_by_side.mp4`**：最大3種類を左から並べた比較動画。並び順は `comparison_order.txt` に記録。
- **`report.json`**：どの設定で、何秒かけて、どれだけ再利用したかの記録。
- **`generation_time.json` / `segments_plan.json`**：H3の所要時間／音声分割の記録。
- **`H3_LatentSync_Results.zip`**：完成MP4などをまとめたZIP。特定の人物名は付けません。

### 注意しておきたいこと

- **音声と動画の「長さの一致」は、口の動きが「発音と完全一致」という意味ではありません。** 正確さは動画を再生して確認します。
- DeepCache・guidance・stepsの変更は、**映像の見た目とのトレードオフ**があります。数値を上げれば必ず良くなるわけではありません。
- 10分の長尺は分割生成を目指していますが、**10分を通した顔・服装・手の連続性はまだ実証されていません**。長尺本番前に30～60秒で確認してください。
- エラーが出た場合は、赤いエラー全文と `latentsync_setup.log` または `ffmpeg_v9.log` などのログを残してください。

---

## ↓ ここから実際に操作します

まずは **①で素材をアップロード**してください。セル名の左側の▶を押して実行します。各入力項目の説明は、入力欄のすぐ上にあります。

> **v13.1 修正（重要）**：⑥の `NameError: name 'chain_dirty' is not defined` を修正しました。同じColabランタイム上で設定と入力が変わらなければ、⑥から再実行するだけで再開できます。途中まで生成できていたクリップは、対応するlatentファイルが揃っていれば再利用します。1本でも再生成すると後続クリップは順次作り直し、古いlatentとの混在を防ぎます。


### 🛠 v14.1 の修正（2026-10-25）

- v14でクリップ2のComfyUI APIが400エラーになる問題を修正。**MultiRef版**の必須設定・`context_frames`（前の動画の末尾連続フレーム）を接続します。
- `H3_CONTEXT_FRAMES=5`、`H3_AUDIO_CONTEXT_FRAMES=18` の場合、音声コンテキストを5フレームに自動制限します。
- GPU生成の前に `/object_info` でノード仕様を検証します。
- 既存のH3クリップとlatentsが有効で、同じジョブ設定・入力画像と音声が一致する場合は、**既存ジョブを自動検索して完成済みクリップを再利用します**。Colabランタイムが初期化されファイルが消えた場合は再利用できません。

**実装上の注意**：このノートはContext Loop拡張を導入しますが、実際の連続生成にはMultiRef Motion Contextを使っています。Context Loop本体のPlan Studio / Review Gate / 自動ループノードへの完全移行ではありません。


In [ ]:
#@title ① ファイルを選ぶ｜画像・音声・既存動画のアップロード
#@markdown **ここでやること**：使いたい写真・イラストと、日本語の元音声を選びます。新規生成では「画像＋音声」、手元のMP4を使うなら「MP4＋音声」です。
#@markdown **おすすめ**：初めてならすべて初期値のまま（画像・音声を別々に選択）。既存MP4なら「作成済みH3動画をアップロード＝ON」「元画像をアップロード＝OFF」にして、②で「既存H3動画から再開」を選びます。
#@markdown **補足**：同じランタイムで再操作するときは「前回のアップロードを使う」をONにすると、`/content/h3_latentsync_workspace/inputs` 内の素材を再利用します。候補が複数ある場合は各種類の最後に更新されたものが選ばれます。
#@markdown ---
#@markdown **前回のアップロードを使う**：ONならこのノートブックの入力フォルダから既存ファイルを選び、アップロード画面をスキップします。初回は **OFF**。※再利用時は以下のアップロード設定は使用しません。
REUSE_PREVIOUS_UPLOADS = False #@param {type:"boolean"}
#@markdown **画像・音声を個別にアップロード**：ONなら別々に選択、OFFならまとめて選択します。おすすめは **ON**。
SPLIT_UPLOADS = True #@param {type:"boolean"}
#@markdown **作成済みH3動画をアップロード**：手元のMP4を使って口パクだけ直す場合は **ON**。新規H3生成／保存済みジョブ利用なら通常はOFF。※②の実行モードも「既存H3動画から再開」にします。
UPLOAD_EXISTING_H3_VIDEO = False #@param {type:"boolean"}
#@markdown **元画像をアップロード**：新しくH3映像を作るなら **ON（必須）**。H3動画を再利用してLatentSyncだけ実行するなら **OFFでもOK**。
UPLOAD_IMAGE = True #@param {type:"boolean"}
from pathlib import Path
from google.colab import files
from IPython.display import display, Audio, Image
import shutil, os
WORK=Path('/content/h3_latentsync_workspace'); UP=WORK/'inputs';UP.mkdir(parents=True,exist_ok=True)
IMAGE_EXT={'.png','.jpg','.jpeg','.webp'}
AUDIO_EXT={'.wav','.mp3','.m4a','.ogg','.flac','.aac'}
VIDEO_EXT={'.mp4','.mov','.mkv','.webm'}

def _save_uploaded(uploaded):
    saved=[]
    for name in uploaded:
        src_path=Path(name)
        target=UP/src_path.name
        if src_path.resolve()!=target.resolve():
            shutil.copy2(src_path,target)
        saved.append(target)
    return saved

candidates=[]
if REUSE_PREVIOUS_UPLOADS:
    candidates=sorted((p for p in UP.iterdir() if p.is_file() and p.stat().st_size>0 and p.suffix.lower() in (IMAGE_EXT|AUDIO_EXT|VIDEO_EXT)),key=lambda p:p.stat().st_mtime_ns)
    print('前回アップロードを再利用:', [p.name for p in candidates])
else:
    if SPLIT_UPLOADS:
        if UPLOAD_IMAGE:
            print('①-1 参照画像をアップロード（新規生成で必要）')
            up_img=files.upload()
            candidates += _save_uploaded(up_img)
        else:
            print('⏭ 画像のアップロードを省略')
        print('①-2 元音声をアップロード（必須）')
        up_aud=files.upload()
        candidates += _save_uploaded(up_aud)
        if UPLOAD_EXISTING_H3_VIDEO:
            print('①-3 既存H3動画をアップロード（再開モードで使用）')
            up_vid=files.upload()
            candidates += _save_uploaded(up_vid)
    else:
        print('画像・音声・必要なら既存H3動画をまとめて選択してください')
        uploaded=files.upload()
        candidates += _save_uploaded(uploaded)

# 直近アップロード優先。複数あれば最後の1個を採用。
imgs=[p for p in candidates if p.suffix.lower() in IMAGE_EXT]
auds=[p for p in candidates if p.suffix.lower() in AUDIO_EXT]
vids=[p for p in candidates if p.suffix.lower() in VIDEO_EXT]
IMAGE_PATH=imgs[-1] if imgs else None
AUDIO_PATH=auds[-1] if auds else None
UPLOADED_H3_VIDEO=vids[-1] if vids else None
assert AUDIO_PATH is not None, '元音声（wav/mp3/m4a等）をアップロードしてください'
print('✅ 元音声:',AUDIO_PATH, 'サイズ(MB):',round(AUDIO_PATH.stat().st_size/1e6,1))
print('✅ 参照画像:',IMAGE_PATH if IMAGE_PATH else 'なし（再開モードなら不要）')
print('✅ 既存H3動画:',UPLOADED_H3_VIDEO if UPLOADED_H3_VIDEO else 'なし（新規生成モードなら不要）')
if IMAGE_PATH: display(Image(filename=str(IMAGE_PATH),width=280))


In [ ]:
#@title ② 何を作るか決める｜動画の長さ・動き・口パク設定
#@markdown **初めてなら、このセルはそのままで実行してOK。** お試しだけなら「試しに生成する長さ」を **15** 秒、本番の全尺なら **0** 秒にします。
#@markdown **費用を抑えるコツ**：口パクだけ直したいときは最初の「実行モード」を **保存済みH3ジョブ** または **既存H3動画** に。比較動画を作る場合は「複数設定の一括比較」をONにします。
#@markdown **読み方**：各入力欄の直前に、用途・目安の値・どの処理に影響するかを説明しています。迷ったら初期値を使ってください。
#@markdown ---
#@markdown ### 🚦 実行方法を選ぶ
#@markdown **実行モード**：①「H3生成から実行」=画像＋音声から新規動画、②「保存済みH3ジョブ」=同じColab内のH3結果を再利用、③「既存H3動画」=アップロード済みMP4を再利用。**口パク調整だけなら②か③**（H3は再生成しません）。
PIPELINE_MODE = 'H3生成から実行' #@param ['H3生成から実行','保存済みH3ジョブから再開','既存H3動画から再開']
REUSE_H3 = not PIPELINE_MODE.startswith('H3生成')
REUSE_SAVED_JOB = PIPELINE_MODE.startswith('保存済み')
REUSE_EXTERNAL_VIDEO = PIPELINE_MODE.startswith('既存H3')
#@markdown **保存済みH3ジョブの場所**：実行モード②で使用。空欄なら保存済みの候補から自動検出。複数ジョブがあり別のものを使いたい場合だけ、ジョブのフォルダを記入します。
#@markdown 例：`/content/h3_latentsync_workspace/outputs/（対象ジョブ名）`。※実行モード①と③では通常不要。
SAVED_JOB_DIR = '' #@param {type:'string'}
if not REUSE_H3:
    assert IMAGE_PATH is not None and IMAGE_PATH.exists(), '新規H3生成には①で画像が必要です'
if REUSE_EXTERNAL_VIDEO:
    assert UPLOADED_H3_VIDEO is not None and UPLOADED_H3_VIDEO.exists(), '外部MP4再開には①で動画アップロードが必要です'
assert AUDIO_PATH is not None and AUDIO_PATH.exists(), '音声のアップロードが必要です'

#@markdown ---
#@markdown ### ⏱ 動画の長さ・分割
#@markdown **試しに生成する長さ（秒）**：`0`は音声の全部、`5`や`15`なら冒頭だけを処理。**比較テストは5～15秒、本番は0**。長さを減らすほどGPU使用時間を抑えられます。
TEST_SECONDS = 0.0 #@param {type:'number'}
#@markdown **1クリップの最大秒数**：長い音声・MP4を何秒ごとに分割するか。**5～15秒の範囲、通常は15**。短いほど細かく分かれ、分割境界が増えます。H3新規生成時は分割構成にも関係します。
MAX_SEGMENT_SECONDS = 15.0 #@param {type:'number'}
#@markdown **足りない映像の静止延長を許可**：既存H3動画が必要な秒数より短い場合、末尾フレームの延長を許可します。**おすすめOFF**（意図しない静止映像を防ぐ）。素材の尺不足を許容するときだけON。
ALLOW_FREEZE_WHEN_REUSING = False #@param {type:'boolean'}

#@markdown ---
#@markdown ### 🎞 MiniMax H3の映像生成設定（新規H3生成時のみ）
#@markdown **15秒超の連続生成モード**：**自動（推奨）**なら元音声が15秒より長い場合だけ、MultiRef Motion Contextによる連続生成を使って前クリップの動きを引き継ぎます。OFFにすると従来の独立生成と単純結合になります。既存H3再利用時は影響しません。
H3_CONTINUITY = '自動（15秒超でContext Loop関連（MultiRef））' #@param ['自動（15秒超でContext Loop関連（MultiRef））','OFF（従来の分割生成）']
#@markdown **引き継ぐ映像のフレーム数**：**5（顔維持寄り）**、**22（滑らかさ寄り）**。大きいほど動きはつながりやすいですが、前クリップへの依存が強くなり、長尺では顔ズレが出やすくなることがあります。長尺のH3生成時のみ有効。
H3_CONTEXT_FRAMES = '5' #@param ['5','22','39','56']
#@markdown **引き継ぐH3音声のフレーム数**：**18（推奨）**＝0.75秒分。H3内部の音声の連続性に使います。完成動画の音声は常にアップロードした元音声なので変更不要です。
H3_AUDIO_CONTEXT_FRAMES = 18 #@param {type:'integer'}
#@markdown **何クリップごとに元画像へ戻すか**：**2（推奨）**なら「2クリップごとにContextを切って再アンカー」します。`0`なら最後まで連続引き継ぎ。**長尺で顔が少しずつ変わる場合は2か3がおすすめ**です。
H3_CONTEXT_RESET_EVERY = 2 #@param {type:'integer'}
#@markdown **映像サイズ／縦横比**：横長はYouTube解説、縦長はShorts向け。基本は**横長 832×480**。大きい解像度はGPU負荷が増える可能性があります。H3再利用モードでは既存映像の解像度を変えません。
CANVAS = '横長 832x480' #@param ['横長 832x480','縦長 480x832','横長 960x544']
#@markdown **H3の生成ステップ数**：**4=速さ優先、8=現在の基準、20=高負荷**。この設定はH3の全身・表情の動きを作る側で、**LatentSyncの口パク推論ステップとは別**です。H3再利用時は影響しません。
STEP_MODE = '8ステップ（音声品質を検証）' #@param ['4ステップ（実測済み・高速）','8ステップ（音声品質を検証）','20ステップ（LoRAなし・低速）']
#@markdown **乱数シード（再現性）**：同じ条件の比較で乱数を固定したいときの番号。**基本42のまま**。H3新規生成とLatentSyncの推論の両方で使うため、変更すると処理結果やキャッシュの再利用に影響します。
SEED = 42 #@param {type:'integer'}
#@markdown **H3への動作指示（英語）**：視線・表情・手振り・カメラの動きなどを指定。現在は**人物・服装・背景を維持し、自然に話す**指示。変更するとH3の映像自体を作り直す必要があります。H3再利用時には適用されません。
MOTION_PROMPT = 'The same adult person from <Picture 1> speaks naturally to the camera. Use <Audio 1> only as a reference for the speech timing. Preserve the person, hairstyle, body, existing clothing and background. Subtle facial expressions, gentle head and hand movements. Fixed medium shot, locked camera, no zoom, no abrupt changes, no new objects, no subtitles.' #@param {type:'string'}

#@markdown ---
#@markdown ### 👄 LatentSyncの口パク設定（通常モード）
#@markdown **口パク推論ステップ数**：口元修正を何ステップ行うか。**20が検証済みの基準**。増やすほど処理が遅くなる傾向があり、精度向上は保証されません。**比較モードONではこの値は使いません**。
LIPSYNC_STEPS = 20 #@param {type:'integer'}
#@markdown **口パク誘導の強さ**：音声に沿った口元生成の誘導係数。**基準1.5**。1.8／2.0も比較候補ですが、上げると顔の歪み・ちらつきが増える場合があります。**比較モードONではこの値は使いません**。
LIPSYNC_GUIDANCE = 1.5 #@param {type:'number'}
#@markdown **DeepCacheによる高速化**：**ON推奨**。LatentSync推論の速度改善を狙う機能。口元の見え方が変わる可能性があるため、厳密に比較するときはON/OFFを別実験にします。通常・比較モードの両方に適用。
LIPSYNC_DEEPCACHE = True #@param {type:'boolean'}

#@markdown ---
#@markdown ### 🔬 複数の口パク設定を比較
#@markdown **複数設定の一括比較**：**ON**にすると次のstepsとguidanceを組み合わせて別々の完成MP4を生成。**OFF**なら上の`LIPSYNC_STEPS`／`LIPSYNC_GUIDANCE`だけを使用。H3映像は共通で使い回します。
COMPARE_LIPSYNC = False #@param {type:'boolean'}
#@markdown **比較する口パクステップ数**：カンマ区切りで指定（例：`20` または `10,20`）。**初回は20だけ**が無難。※H3の`STEP_MODE`とは別物。比較モードOFFなら使用しません。
COMPARE_STEPS = '20' #@param {type:'string'}
#@markdown **比較する誘導係数**：カンマ区切りで指定（例：`1.5,1.8,2.0`）。20ステップ×3値なら**3本のMP4**を生成。steps×guidanceは最大9通り。値が増えるほどGPU時間も増えます。
COMPARE_GUIDANCES = '1.5,1.8,2.0' #@param {type:'string'}

#@markdown ---
#@markdown ### 📦 比較動画とファイル出力
#@markdown **横並び比較動画を作る**：ONなら最大3種類を横に並べた比較MP4を作成。**比較するならON**。LatentSync推論数は変わらず、比較動画の合成処理が追加されます。
CREATE_SIDE_BY_SIDE = True #@param {type:'boolean'}
#@markdown **元H3素材をZIPにも保存**：ONなら最終MP4のほかにH3の元クリップもZIPに含めます。ZIPが大きくなるので、**普段はOFF推奨**。元H3はColabの作業フォルダには残ります。
INCLUDE_RAW_H3_IN_ZIP = False #@param {type:'boolean'}
#@markdown **ZIPを自動ダウンロード**：ONなら最後にブラウザのダウンロードを開始。**通常はOFFでもOK**（保存先のZIPから手動取得できます）。大容量ZIPではブラウザ側で失敗することがあります。
AUTO_DOWNLOAD_ZIP = False #@param {type:'boolean'}

import math, json, subprocess
from pathlib import Path
NUM_STEPS=int(STEP_MODE.split('ステップ')[0]); USE_TURBO=NUM_STEPS in (4,8)
WIDTH,HEIGHT={'横長 832x480':(832,480),'縦長 480x832':(480,832),'横長 960x544':(960,544)}[CANVAS]
assert WIDTH%32==0 and HEIGHT%32==0
assert TEST_SECONDS==0 or TEST_SECONDS>=1.0
assert 5<=MAX_SEGMENT_SECONDS<=15
assert H3_CONTEXT_RESET_EVERY>=0

def source_duration(path):
    return float(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration',
      '-of','default=noprint_wrappers=1:nokey=1',str(path)],text=True).strip())

SOURCE_AUDIO_DURATION=source_duration(AUDIO_PATH)
TARGET_AUDIO_SECONDS=min(SOURCE_AUDIO_DURATION,float(TEST_SECONDS)) if TEST_SECONDS else SOURCE_AUDIO_DURATION
assert TARGET_AUDIO_SECONDS>=1.0
H3_USE_CONTEXT = (not REUSE_H3 and H3_CONTINUITY.startswith('自動') and TARGET_AUDIO_SECONDS > MAX_SEGMENT_SECONDS)
H3_CONTEXT_FRAMES_INT = int(H3_CONTEXT_FRAMES)
assert H3_CONTEXT_FRAMES_INT in (5,22,39,56)
assert 0 <= H3_AUDIO_CONTEXT_FRAMES <= 240

def h3_aligned_frames(seconds):
    return 5 + 17*max(0,math.ceil((seconds*24-5-1e-8)/17))
H3_MAX_FRAMES=h3_aligned_frames(MAX_SEGMENT_SECONDS)
H3_FIRST_CAP_25=min(math.floor(MAX_SEGMENT_SECONDS*25+1e-7),
                    math.floor((H3_MAX_FRAMES /24)*25 - 1e-6))
H3_NEXT_CAP_25=min(math.floor(MAX_SEGMENT_SECONDS*25+1e-7),
                   math.floor(((H3_MAX_FRAMES-H3_CONTEXT_FRAMES_INT)/24)*25-1e-6)) if H3_USE_CONTEXT else H3_FIRST_CAP_25
assert H3_NEXT_CAP_25>=25, '長尺コンテキストが長すぎます。最大秒数を長くするか引継ぎを短くしてください。'
H3_TARGET_FRAMES_25=math.ceil(TARGET_AUDIO_SECONDS*25-1e-6)

def is_fresh_anchor_segment(seg_zero_idx:int)->bool:
    if seg_zero_idx==0:
        return True
    if not H3_USE_CONTEXT:
        return True
    if H3_CONTEXT_RESET_EVERY<=0:
        return False
    return (seg_zero_idx % H3_CONTEXT_RESET_EVERY)==0

def segment_capacity_25(seg_zero_idx:int)->int:
    return H3_FIRST_CAP_25 if is_fresh_anchor_segment(seg_zero_idx) else H3_NEXT_CAP_25

EXPECTED_SEGMENTS=0
frames_accum=0
while frames_accum < H3_TARGET_FRAMES_25:
    frames_accum += segment_capacity_25(EXPECTED_SEGMENTS)
    EXPECTED_SEGMENTS += 1

def list_numbers(s,kind):
    cast=int if kind=='step' else float
    values=[cast(t.strip()) for t in s.split(',') if t.strip()]
    assert values, '比較用の設定リストが空です'
    return list(dict.fromkeys(values))

def validate_variant(steps,guidance):
    assert 4<=int(steps)<=50, f'steps={steps} は範囲外(4～50)'
    assert 1.0<=float(guidance)<=3.0, f'guidance={guidance} は範囲外(1～3)'

if COMPARE_LIPSYNC:
    step_list=list_numbers(COMPARE_STEPS,'step'); guidance_list=list_numbers(COMPARE_GUIDANCES,'guidance')
else:
    step_list=[int(LIPSYNC_STEPS)]; guidance_list=[float(LIPSYNC_GUIDANCE)]
VARIANTS=[]
for st in step_list:
    for gs in guidance_list:
        validate_variant(st,gs)
        tag=f's{st}_g{gs:g}_dc{int(LIPSYNC_DEEPCACHE)}'.replace('.','p')
        VARIANTS.append({'steps':st,'guidance':gs,'deepcache':bool(LIPSYNC_DEEPCACHE),'tag':tag})
assert len(VARIANTS)<=9, '比較設定は最大9通り。短いテストから試してください'

print(f'🎙 {SOURCE_AUDIO_DURATION:.3f}秒 → 対象{TARGET_AUDIO_SECONDS:.3f}秒 / 予定H3クリップ数 {EXPECTED_SEGMENTS}')
if H3_USE_CONTEXT:
    reset_text='最後まで連続引継ぎ' if H3_CONTEXT_RESET_EVERY<=0 else f'{H3_CONTEXT_RESET_EVERY}クリップごとに元画像へ再アンカー'
    print(f'🔗 連続生成: Context Loop関連（MultiRef） / context={H3_CONTEXT_FRAMES_INT}f / audio_context={H3_AUDIO_CONTEXT_FRAMES}f / {reset_text}')
else:
    print('🎬 分割方式: 従来の独立生成（15秒以下、または連続生成OFF）')
print('👄 LatentSync:', ', '.join(f"{v['tag']}" for v in VARIANTS))


In [ ]:
#@title ③【新規生成のみ】H3を使うための準備
#@markdown **このセルの役割**：MiniMax H3を動かすプログラムを用意します。15秒超で連続生成する場合は **Context Loop関連ノード** と **MultiRef Motion Context** もインストールします。
#@markdown **ポイント**：②で既存動画から再開を選んだ場合、このセルを実行しても自動でスキップされます。
#@markdown ---
if REUSE_H3:
    print("⏭ 既存H3動画を使うため、このセルをスキップします。")
else:
    import os, subprocess, sys, time, json, shutil
    from pathlib import Path
    COMFY = Path('/content/ComfyUI')

    def run(cmd, cwd=None):
        print('▶', ' '.join(map(str,cmd)), flush=True)
        subprocess.run(list(map(str,cmd)),cwd=str(cwd) if cwd else None,check=True)

    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True))
    import torch
    assert torch.cuda.is_available(), 'GPUランタイムに接続してください'
    print('torch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))
    disk_free_gib=shutil.disk_usage('/content').free/1024**3
    print(f'空きディスク: {disk_free_gib:.1f} GiB')
    if disk_free_gib < 45:
        print('⚠ 空きディスクが45GiB未満です。モデルが取得済みならそのまま再利用可能ですが、新規ダウンロード時は容量不足に注意してください。')
    if not (COMFY/'main.py').exists():
        run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git',COMFY])
    run([sys.executable,'-m','pip','install','-q','-r',COMFY/'requirements.txt'])
    run([sys.executable,'-m','pip','install','-q','-U','huggingface_hub[hf_xet]','requests','pillow'])
    run(['apt-get','update','-qq'])
    run(['apt-get','install','-y','-qq','ffmpeg'])

    custom=COMFY/'custom_nodes'
    custom.mkdir(parents=True,exist_ok=True)
    if H3_USE_CONTEXT:
        legacy=custom/'ComfyUI-H3-Motion-Context'
        if legacy.exists():
            print('🧹 旧H3 Motion Contextを削除（MultiRef版へ置換）')
            shutil.rmtree(legacy, ignore_errors=True)
        multiref=custom/'ComfyUI-H3-Motion-Context-MultiRef'
        if not multiref.exists():
            run(['git','clone','--depth','1',
                 'https://github.com/seitanism/ComfyUI-H3-Motion-Context-MultiRef.git', multiref])
        ctxloop=custom/'ComfyUI-MiniMaxH3-Context-Loop'
        if not ctxloop.exists():
            run(['git','clone','--depth','1','--branch','main',
                 'https://github.com/ethanfel/ComfyUI-MiniMaxH3-Context-Loop.git', ctxloop])
        for req in [multiref/'requirements.txt', ctxloop/'requirements.txt']:
            if req.exists():
                run([sys.executable,'-m','pip','install','-q','-r',req])
        print('✅ Context Loop関連ノードを準備しました')
    else:
        print('ℹ 連続生成OFFのため、Context Loop関連ノードは不要です。')


In [ ]:
#@title ④【新規生成のみ】H3のAIモデルを準備
#@markdown **このセルの役割**：表情や体の動きを作るための大きなモデルファイルをダウンロードします。初回は時間とディスク容量が必要です。
#@markdown **ポイント**：同じランタイムにモデルが残っていれば取得済みファイルを利用します。既存動画から再開ならスキップされます。
#@markdown ---
if REUSE_H3:
    print("⏭ 既存H3動画を使うため、このセルをスキップします。")
else:
    #@markdown 通常はこのセルが最もダウンロード時間を使います。中断後の再実行ではHugging Faceが取得済みを再利用します。
    from huggingface_hub import hf_hub_download
    from pathlib import Path
    import shutil, time
    REPO = 'Comfy-Org/MiniMax-H3'
    MODEL_FILES = [
      'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors',
      'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
      'vae/minimax_h3_video_vae_int8_convrot.safetensors',
      'vae/minimax_h3_audio_vae_fp32.safetensors',
    ]
    LORA_NAME = (
        'minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors' if NUM_STEPS == 4 else
        'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors' if NUM_STEPS == 8 else None
    )
    # 4-stepは前回と同じモデルを使う。8-stepは別の専用LoRAが必要。
    if NUM_STEPS == 4:
      MODEL_FILES.append('loras/'+LORA_NAME)
    start_dl=time.perf_counter()
    for filename in MODEL_FILES:
        dest=COMFY/'models'/filename
        dest.parent.mkdir(parents=True,exist_ok=True)
        if dest.exists() and dest.stat().st_size>100*1024*1024:
            print('✅ 再利用:', filename, f'{dest.stat().st_size/2**30:.2f}GiB')
            continue
        print('⬇ ダウンロード:',filename,flush=True)
        hf_hub_download(repo_id=REPO,filename=filename,local_dir=str(COMFY/'models'))
        assert dest.exists() and dest.stat().st_size>100*1024*1024, f'モデルが不完全: {filename}'
        print('✅',f'{dest.stat().st_size/2**30:.2f}GiB',flush=True)
    if NUM_STEPS == 8:
        target = COMFY / 'models' / 'loras' / LORA_NAME
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.stat().st_size > 100*1024*1024:
            print('✅ 8ステップRef2VA LoRAを再利用:', target.name)
        else:
            print('⬇ 8ステップ専用LoRAを取得（約2GB）:', LORA_NAME, flush=True)
            hf_hub_download(repo_id='lightx2v/Minimax-h3-Turbo', filename=LORA_NAME,
                            local_dir=str(COMFY/'models'/'loras'))
            assert target.exists() and target.stat().st_size > 100*1024*1024, '8ステップLoRA取得に失敗'
    print(f'モデル準備時間：{(time.perf_counter()-start_dl)/60:.1f}分')


In [ ]:
#@title ⑤【新規生成のみ】H3の生成サーバーを起動
#@markdown **このセルの役割**：H3が動画を作るためのComfyUIを起動し、必要な機能が使えるか確かめます。
#@markdown **ポイント**：このセルが終わってから⑥でH3動画を生成します。口パクだけ直すときは不要です。
#@markdown ---
if REUSE_H3:
    print("⏭ 既存H3動画を使うため、このセルをスキップします。")
else:
    import subprocess, os, json, time, requests
    LOG = WORK/'comfyui.log'
    URL='http://127.0.0.1:8188'
    try:
        r=requests.get(URL+'/system_stats',timeout=2)
        online=r.ok
    except Exception:
        online=False
    if not online:
        cmd=[sys.executable,str(COMFY/'main.py'),'--listen','127.0.0.1','--port','8188','--disable-auto-launch','--disable-pinned-memory']
        # 48GB未満のGPUではモデルをシステムRAMへオフロードしてOOMリスクを軽減
        if torch.cuda.get_device_properties(0).total_memory / 1024**3 < 48:
            cmd.append('--lowvram')
            print('⚠ 48GB未満のGPUなので --lowvram で起動します')
        env=os.environ.copy()
        env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
        log_handle=open(LOG,'a',encoding='utf-8')
        COMFY_PROCESS=subprocess.Popen(cmd,cwd=str(COMFY),stdout=log_handle,stderr=subprocess.STDOUT,env=env,start_new_session=True)
        print('ComfyUI起動中…')
        for i in range(180):
            if COMFY_PROCESS.poll() is not None:
                raise RuntimeError('ComfyUIが終了しました。'+str(LOG)+' の末尾を確認してください')
            try:
                if requests.get(URL+'/system_stats',timeout=3).ok:
                    online=True
                    break
            except Exception: pass
            time.sleep(2)
    assert online,'ComfyUI起動に時間がかかっています。comfyui.logをご確認ください'
    objects=requests.get(URL+'/object_info',timeout=60).json()
    required=['UNETLoader','CLIPLoader','VAELoader','MiniMaxH3ReferenceToVideo',
              'LoadImage','LoadAudio','BasicGuider','KSamplerSelect','BasicScheduler',
              'RandomNoise','SamplerCustomAdvanced','VAEDecode','VAEDecodeAudio',
              'CreateVideo','SaveVideo']
    if USE_TURBO: required.append('LoraLoaderModelOnly')
    if NUM_STEPS == 8: required.append('MiniMaxH3SigmaShift')
    if H3_USE_CONTEXT:
        required += ['MiniMaxH3MotionContext', 'MiniMaxH3MotionContextTrim',
                     'MiniMaxH3MotionContextSaveLatent','MiniMaxH3MotionContextLoadLatent']
    missing=[x for x in required if x not in objects]
    assert not missing,'ComfyUIに必要なノードがありません: '+', '.join(missing)+'。ComfyUIの更新が必要です。'
    print('✅ ComfyUI起動・必要ノードの存在確認が成功')
    if H3_USE_CONTEXT: print('✅ Motion Context用ノードが利用できます（拡張機能の内部契約チェックは初回生成時に実施）。')
    print('場所:',URL,'\nログ:',LOG)


In [ ]:
#@title ⑥【GPU使用】H3で表情・体・手が動く動画を作る
#@markdown **このセルの役割**：人物画像と元音声を参考に、話している人物の映像を作成します。15秒より長ければ、**MultiRef Motion Contextによる連続生成**で前クリップの動きを引き継ぎます。
#@markdown **ポイント**：**GPU時間を多く使う工程**です。v14.1ではMultiRef版の必須入力を適切に接続します。前のH3生成ジョブが同じランタイムに残っていれば1本目は再利用できます。
#@markdown ---
if REUSE_H3:
    print('⏭ 既存H3動画から再開モード：⑦-Bへ進んでください。')
else:
    import os, json, math, time, shutil, subprocess, requests, uuid, wave, hashlib
    from pathlib import Path
    from datetime import datetime
    assert IMAGE_PATH.exists() and AUDIO_PATH.exists()
    assert all((COMFY/'models'/x).exists() for x in MODEL_FILES), '先に④モデル取得'
    assert requests.get(URL+'/system_stats',timeout=15).ok, '先に⑤ComfyUIを起動'

    def ff(cmd):
        subprocess.run(cmd,check=True)
    def probe_duration(path):
        return float(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration',
            '-of','default=noprint_wrappers=1:nokey=1',str(path)],text=True).strip())
    def aligned_h3_frames(seconds):
        return 5 + 17*max(0,math.ceil((seconds*24-5-1e-8)/17))
    def seg_is_fresh(seg_zero_idx:int)->bool:
        if seg_zero_idx==0:
            return True
        if not H3_USE_CONTEXT:
            return True
        if H3_CONTEXT_RESET_EVERY<=0:
            return False
        return (seg_zero_idx % H3_CONTEXT_RESET_EVERY)==0
    def seg_overlap(seg_zero_idx:int)->int:
        return 0 if seg_is_fresh(seg_zero_idx) else H3_CONTEXT_FRAMES_INT
    def seg_capacity_25(seg_zero_idx:int)->int:
        return H3_FIRST_CAP_25 if seg_is_fresh(seg_zero_idx) else H3_NEXT_CAP_25

    image_id=(str(IMAGE_PATH),IMAGE_PATH.stat().st_size,IMAGE_PATH.stat().st_mtime_ns)
    audio_id=(str(AUDIO_PATH),AUDIO_PATH.stat().st_size,AUDIO_PATH.stat().st_mtime_ns)
    config_id=json.dumps([image_id,audio_id,TEST_SECONDS,MAX_SEGMENT_SECONDS,WIDTH,HEIGHT,
                          NUM_STEPS,SEED,MOTION_PROMPT,H3_USE_CONTEXT,H3_CONTEXT_FRAMES_INT,
                          H3_AUDIO_CONTEXT_FRAMES,H3_CONTEXT_RESET_EVERY],ensure_ascii=False)
    # v14からの途中再開：同一ランタイムで別ノートブックを開いてもジョブを検索する。
    # ファイル名のみ・音声尺のみで誤って復元しないよう、設定と入力ファイル実体を検証。
    def sha256_file(path):
        h=hashlib.sha256()
        with open(path,'rb') as f:
            for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
        return h.hexdigest()
    current_cfg=json.loads(config_id)
    def file_matches_old(old_spec, current_path):
        try:
            old_path=Path(old_spec[0]); st=old_path.stat()
            # 旧ジョブ作成時のファイルが上書きされていたら使い回し禁止
            if st.st_size!=int(old_spec[1]) or st.st_mtime_ns!=int(old_spec[2]):
                return False
            current_path=Path(current_path)
            if old_path.resolve()==current_path.resolve():
                return True
            if st.st_size!=current_path.stat().st_size:
                return False
            return sha256_file(old_path)==sha256_file(current_path)
        except (OSError,ValueError,TypeError,IndexError):
            return False
    def can_resume_job(candidate):
        try:
            cfg_file=Path(candidate)/'job_config.txt'
            old_cfg=json.loads(cfg_file.read_text())
            if old_cfg==current_cfg:
                return True
            # 最初の2要素はアップロード時のパスとmtime。中身同一なら別パスでも再開。
            if old_cfg[2:]!=current_cfg[2:]:
                return False
            return (file_matches_old(old_cfg[0],IMAGE_PATH)
                    and file_matches_old(old_cfg[1],AUDIO_PATH))
        except (OSError,ValueError,TypeError,IndexError,json.JSONDecodeError):
            return False
    old_job=globals().get('JOB')
    root=WORK/'outputs'
    candidates=[Path(old_job)] if old_job else []
    if root.is_dir():
        candidates+=sorted((c for c in root.iterdir() if c.is_dir()),
                           key=lambda c:c.stat().st_mtime_ns, reverse=True)
    JOB=next((c for c in candidates if can_resume_job(c)),None)
    if JOB is not None:
        JOB=Path(JOB)
        print('♻ 既存のH3生成ジョブを再開:',JOB)
    else:
        JOB=root/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        JOB.mkdir(parents=True,exist_ok=True)
        (JOB/'job_config.txt').write_text(config_id)
    SEG_DIR=JOB/'segments';SEG_DIR.mkdir(parents=True,exist_ok=True)
    MASTER=JOB/'source_mono_16k.wav'
    if not MASTER.exists():
        ff(['ffmpeg','-y','-hide_banner','-loglevel','error',
            '-i',str(AUDIO_PATH),'-t',f'{TARGET_AUDIO_SECONDS:.6f}',
            '-ac','1','-ar','16000','-c:a','pcm_s16le',str(MASTER)])
    with wave.open(str(MASTER),'rb') as wf:
        assert wf.getframerate()==16000 and wf.getnchannels()==1 and wf.getsampwidth()==2
        sample_count=wf.getnframes(); pcm=wf.readframes(sample_count)
    from array import array
    samples=array('h'); samples.frombytes(pcm)
    if not samples: raise RuntimeError('音声が空です')
    actual_audio_sec=sample_count/16000
    total_frames=math.ceil(actual_audio_sec*25-1e-6)

    capacities=[]; cap_sum=0
    while cap_sum < total_frames:
        cap=seg_capacity_25(len(capacities))
        capacities.append(cap); cap_sum += cap
    chunk_count=len(capacities)

    boundaries=[0]
    def energy(frame):
        ix=frame*640
        seg=samples[max(0,ix-1280):min(sample_count,ix+1280)]
        return sum((x/32768)**2 for x in seg[::8]) / max(1,len(seg[::8]))

    for i in range(chunk_count-1):
        at=boundaries[-1]
        remaining=total_frames-at
        future_caps=capacities[i+1:]
        future_min=25*len(future_caps)
        current_cap=capacities[i]
        min_len=max(25, remaining-sum(future_caps))
        max_len=min(current_cap, remaining-future_min)
        if min_len>max_len: raise RuntimeError('分割計画の容量が足りません')
        ideal=round(remaining/(len(future_caps)+1))
        lo=max(min_len,ideal-15); hi=min(max_len,ideal+15)
        if lo>hi: lo,hi=min_len,max_len
        best=min(range(lo,hi+1), key=lambda length:(energy(at+length)+0.000035*abs(length-ideal), abs(length-ideal)))
        boundaries.append(at+best)
    boundaries.append(total_frames)

    plan=[]
    for i,(a,b) in enumerate(zip(boundaries[:-1],boundaries[1:])):
        frames_25=b-a
        seg_len=frames_25/25
        overlap=seg_overlap(i)
        h3_frames=aligned_h3_frames(seg_len+overlap/24)
        assert h3_frames-overlap>=math.ceil(frames_25*24/25-1e-6), '継続生成フレーム数が不足'
        assert h3_frames<=H3_MAX_FRAMES, f'セグメント{i+1}が設定最大{H3_MAX_FRAMES}fを超えました: {h3_frames}f'
        seg_path=SEG_DIR/f'segment_{i+1:03d}'
        seg_path.mkdir(exist_ok=True)
        ref=seg_path/'ref_audio_32k.wav'
        if not ref.exists():
            raw=samples[a*640:min(b*640,sample_count)]
            padding=max(0,frames_25*640-len(raw))
            with wave.open(str(seg_path/'mono_16k.wav'),'wb') as wav:
                wav.setnchannels(1); wav.setframerate(16000); wav.setsampwidth(2)
                wav.writeframes(raw.tobytes()+bytes(padding*2))
            ff(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(seg_path/'mono_16k.wav'),
                '-ar','32000','-ac','2','-c:a','pcm_s16le',str(ref)])
        plan.append({'index':i+1,'start_frame_25':a,'end_frame_25':b,
            'start_sec':a/25,'end_sec':b/25,'duration':seg_len,
            'frames_25':frames_25,'h3_frames':h3_frames,'h3_duration':h3_frames/24,
            'context_trim_frames':overlap,'continuity_mode':'context_loop' if overlap>0 else 'fresh_anchor',
            'h3_continuity':bool(H3_USE_CONTEXT),'ref_audio':str(ref),
            'h3_video':str(seg_path/'MiniMax_H3_generated.mp4'),'segment_dir':str(seg_path)})
    PLAN=plan
    (JOB/'segments_plan.json').write_text(json.dumps(PLAN,ensure_ascii=False,indent=2))
    print('\n📋 セグメント計画')
    for x in PLAN:
        mode='Context Loop' if x['context_trim_frames']>0 else 'Fresh anchor'
        print(f'  #{x["index"]:02d} 元音声{x["duration"]:.3f}秒 ({x["start_sec"]:.2f}-{x["end_sec"]:.2f}) → H3 {x["h3_frames"]}f / 引継ぎ削除{x["context_trim_frames"]}f / {mode}')
    print(f'🔊 元音声 {actual_audio_sec:.3f}秒 / 最終動画の25fpsフレーム数 {total_frames} / {total_frames/25:.3f}秒')
    if len(PLAN)>1:
        if H3_USE_CONTEXT:
            reset_text='最後まで連続引継ぎ' if H3_CONTEXT_RESET_EVERY<=0 else f'{H3_CONTEXT_RESET_EVERY}クリップごとに元画像へ再アンカー'
            print(f'🔗 Context Loop互換で連続生成します。context={H3_CONTEXT_FRAMES_INT}f / {reset_text}')
        else:
            print('⚠ 従来方式：各クリップは独立生成のため境目が不連続になることがあります。')

    if USE_TURBO:
        lf=COMFY/'models'/'loras'/LORA_NAME
        assert lf.exists() and lf.stat().st_size>100*1024*1024, f'④でLoRAを用意してください: {lf}'
    inp=COMFY/'input'; inp.mkdir(parents=True,exist_ok=True)
    staged_img=inp/f'h3_{uuid.uuid4().hex[:10]}{IMAGE_PATH.suffix}'
    shutil.copy2(IMAGE_PATH, staged_img)

    def stage_previous_tail_frames(previous_video, clip_index, count):
        """MultiRef の必須 IMAGE バッチに渡すため、前クリップ末尾の24fpsフレームを用意。"""
        previous_video=Path(previous_video)
        assert previous_video.is_file(), f'前クリップの動画がありません: {previous_video}'
        raw=subprocess.check_output([
            'ffprobe','-v','error','-select_streams','v:0','-count_frames',
            '-show_entries','stream=nb_read_frames',
            '-of','default=noprint_wrappers=1:nokey=1',str(previous_video)],text=True).strip()
        assert raw.isdigit(), f'前クリップのフレーム数を読めません: {raw!r}'
        total=int(raw)
        assert total>count, f'前クリップ{total}フレームより引継ぎ{count}フレームが長すぎます'
        target_dir=inp/f'h3_context_tail_{JOB.name}_{clip_index:03d}'
        target_dir.mkdir(parents=True, exist_ok=True)
        paths=[target_dir/f'frame_{j:04d}.png' for j in range(count)]
        # 同じ入力に対してのみ再利用。既存コンテキストが古ければ生成し直す。
        manifest=target_dir/'source.json'
        file_identity={'path':str(previous_video),'size':previous_video.stat().st_size,
                       'mtime_ns':previous_video.stat().st_mtime_ns,'frames':total,'tail':count}
        reuse=(manifest.is_file() and all(p.is_file() for p in paths))
        if reuse:
            try: reuse=(json.loads(manifest.read_text())==file_identity)
            except (OSError,json.JSONDecodeError): reuse=False
        if not reuse:
            for p in target_dir.glob('frame_*.png'):p.unlink()
            first=total-count
            ff(['ffmpeg','-y','-hide_banner','-loglevel','error',
                '-i',str(previous_video),'-vf',f'select=gte(n\\,{first})',
                '-fps_mode:v','passthrough','-start_number','0',
                str(target_dir/'frame_%04d.png')])
            produced=sorted(target_dir.glob('frame_*.png'))
            assert len(produced)==count, (
                f'前クリップの末尾画像取得が{len(produced)}/{count}枚で失敗。'
                f' ffprobe={total}フレーム / {previous_video}')
            manifest.write_text(json.dumps(file_identity,ensure_ascii=False,indent=2))
        return [f'{target_dir.name}/{p.name}' for p in paths]

    def build_workflow(ref_filename,frames,index,use_context_this_clip,context_images=None):
        wf={
        '6':{'class_type':'UNETLoader','inputs':{'unet_name':'minimax_h3_ref2va_pruned_int8_convrot.safetensors','weight_dtype':'default'}},
        '13':{'class_type':'CLIPLoader','inputs':{'clip_name':'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors','type':'minimax','device':'default'}},
        '11':{'class_type':'VAELoader','inputs':{'vae_name':'minimax_h3_video_vae_int8_convrot.safetensors'}},
        '24':{'class_type':'VAELoader','inputs':{'vae_name':'minimax_h3_audio_vae_fp32.safetensors'}},
        '30':{'class_type':'LoadImage','inputs':{'image':staged_img.name,'upload':'image'}},
        '60':{'class_type':'LoadAudio','inputs':{'audio':ref_filename}},
        '104':{'class_type':'MiniMaxH3ReferenceToVideo','inputs':{'clip':['13',0],'vae':['11',0],
            'audio_vae':['24',0],'prompt':MOTION_PROMPT,'width':WIDTH,'height':HEIGHT,
            'length':frames,'ref_image_size':'match','ref_images.ref_image_0':['30',0],
            'ref_audios.ref_audio_0':['60',0]}},
        '17':{'class_type':'KSamplerSelect','inputs':{'sampler_name':'euler' if NUM_STEPS==8 else 'res_multistep'}},
        '15':{'class_type':'RandomNoise','inputs':{'noise_seed':SEED+index-1}},
        '10':{'class_type':'VAEDecode','inputs':{'samples':['14',0],'vae':['11',0]}},
        '23':{'class_type':'VAEDecodeAudio','inputs':{'samples':['14',0],'vae':['24',0]}},
        '91':{'class_type':'CreateVideo','inputs':{'images':['10',0],'audio':['23',0],'fps':24}},
        '92':{'class_type':'SaveVideo','inputs':{'video':['91',0],
            'filename_prefix':f'H3_job_{JOB.name}_seg_{index:03d}','format':'auto','codec':'auto'}}}
        if USE_TURBO:
            wf['7']={'class_type':'LoraLoaderModelOnly','inputs':{'model':['6',0],'lora_name':LORA_NAME,'strength_model':1.0}}
        model_ref=['7',0] if USE_TURBO else ['6',0]
        if NUM_STEPS==8:
            wf['8']={'class_type':'MiniMaxH3SigmaShift','inputs':{'model':model_ref,'shift_video':6.,'shift_audio':3.}}
            model_ref=['8',0]
        wf.update({
          '16':{'class_type':'BasicGuider','inputs':{'model':model_ref,'conditioning':['104',0]}},
          '9':{'class_type':'BasicScheduler','inputs':{'model':model_ref,'scheduler':'simple','steps':NUM_STEPS,'denoise':1.0}},
          '14':{'class_type':'SamplerCustomAdvanced','inputs':{'noise':['15',0],
              'guider':['16',0],'sampler':['17',0],'sigmas':['9',0],'latent_image':['104',1]}}})
        if H3_USE_CONTEXT:
            save_prefix=f'h3_context_loop/{JOB.name}/clip'
            wf['108']={'class_type':'MiniMaxH3MotionContextSaveLatent',
                'inputs':{'latent':['14',0], 'filename_prefix':save_prefix, 'clip_index':index}}
            if use_context_this_clip:
                assert context_images is not None and len(context_images)==H3_CONTEXT_FRAMES_INT, (
                    'MultiRefのcontext_frames用に前動画の末尾画像が必要です')
                wf['105']={'class_type':'MiniMaxH3MotionContextLoadLatent',
                    'inputs':{'latent_path':f'h3_context_loop/{JOB.name}',
                              'clip_index':index-1}}
                # MultiRef版では context_frames は必須 IMAGE であり、AV latent は音声側の
                # 連続性に使われる。画像を1枚に潰さず、動画から連続フレームを抜き出して接続。
                batch_ref=None
                for j, image_name in enumerate(context_images):
                    img_id=str(200+j)
                    wf[img_id]={'class_type':'LoadImage',
                                'inputs':{'image':image_name,'upload':'image'}}
                    if batch_ref is None:
                        batch_ref=[img_id,0]
                    else:
                        batch_id=str(300+j)
                        wf[batch_id]={'class_type':'ImageBatch',
                                      'inputs':{'image1':batch_ref,'image2':[img_id,0]}}
                        batch_ref=[batch_id,0]
                effective_audio_frames=min(int(H3_AUDIO_CONTEXT_FRAMES),int(H3_CONTEXT_FRAMES_INT))
                wf['106']={'class_type':'MiniMaxH3MotionContext',
                    'inputs':{'conditioning':['104',0], 'vae':['11',0], 'latent':['104',1],
                              'context_frames':batch_ref,
                              'context_length':int(H3_CONTEXT_FRAMES_INT),
                              'encode_mode':'video','anchor_mode':'head','crop':'disabled',
                              'audio_context_length':effective_audio_frames,
                              'audio_mode':'timeline','target_start':0,
                              'context_latent':['105',0]}}
                wf['16']['inputs']['conditioning']=['106',0]
                wf['107']={'class_type':'MiniMaxH3MotionContextTrim',
                    'inputs':{'images':['10',0], 'audio':['23',0],
                              'trim_frames':['106',1],'fps':24.0,'match_tail':True}}
                wf['91']['inputs']['images']=['107',0]
                wf['91']['inputs']['audio']=['107',1]
        return wf

    chain_dirty=False
    h3_start=time.perf_counter()
    for x in PLAN:
        dst=Path(x['h3_video']); seg=Path(x['segment_dir']); idx=x['index']
        use_context_this_clip = x['context_trim_frames'] > 0
        if not use_context_this_clip:
            chain_dirty=False
        latent_file=COMFY/'output'/'h3_context_loop'/JOB.name/f'clip_{idx:05d}.safetensors'
        has_mp4=dst.exists() and dst.stat().st_size>10000
        has_latent=(latent_file.exists() and latent_file.stat().st_size>10000) if H3_USE_CONTEXT else True
        if H3_USE_CONTEXT and chain_dirty and use_context_this_clip:
            has_mp4=False; has_latent=False
        if has_mp4 and has_latent:
            print(f'♻ H3 #{idx} の生成済みファイル'+('＋latent' if H3_USE_CONTEXT else '')+'を再利用')
            continue
        if use_context_this_clip:
            previous=COMFY/'output'/'h3_context_loop'/JOB.name/f'clip_{idx-1:05d}.safetensors'
            assert previous.is_file() and previous.stat().st_size>10000, f'直前のH3潜在データがありません: {previous}'
        if has_mp4 and not has_latent:
            print(f'⚠ #{idx}: MP4はありますが続き生成用latentがないため、このクリップを再生成します')
        chain_dirty=True if H3_USE_CONTEXT else False
        if H3_USE_CONTEXT and use_context_this_clip:
            print('🔁 このクリップ以降のContext Loop系クリップは、直前latent更新のため順次作り直します')
        elif H3_USE_CONTEXT and not use_context_this_clip and idx>1:
            print('🪝 このクリップは元画像から再アンカーして生成します')
        context_images=None
        if use_context_this_clip:
            prev_mp4=Path(PLAN[idx-2]['h3_video'])
            context_images=stage_previous_tail_frames(prev_mp4,idx,H3_CONTEXT_FRAMES_INT)
            if H3_AUDIO_CONTEXT_FRAMES>H3_CONTEXT_FRAMES_INT:
                print(f'ℹ 音声Contextは{H3_AUDIO_CONTEXT_FRAMES}fから{H3_CONTEXT_FRAMES_INT}fへ自動調整（映像Contextを超えないため）')
            print(f'🖼 直前クリップから末尾{len(context_images)}フレームを抽出: {prev_mp4.name}')
        staged_audio=inp/f'h3_{uuid.uuid4().hex[:10]}.wav'
        shutil.copy2(x['ref_audio'], staged_audio)
        wf=build_workflow(staged_audio.name, x['h3_frames'], idx, use_context_this_clip,context_images)
        if use_context_this_clip:
            # GPU推論前に API の最新スキーマと照合。古いノードとの食い違いを明示。
            defs=requests.get(URL+'/object_info',timeout=60).json()
            required={
                'MiniMaxH3MotionContext':{'conditioning','vae','latent','context_frames',
                    'context_length','encode_mode','anchor_mode','crop','audio_context_length',
                    'audio_mode','target_start'},
                'MiniMaxH3MotionContextLoadLatent':{'latent_path','clip_index'},
                'MiniMaxH3MotionContextTrim':{'images','trim_frames'},
                'ImageBatch':{'image1','image2'},'LoadImage':{'image'},
            }
            for class_type,needed in required.items():
                info=defs.get(class_type)
                assert info, f'ComfyUIが{class_type}を認識しません。③→⑤の再実行とログを確認してください'
                available=set(info.get('input',{}).get('required',{}))|set(info.get('input',{}).get('optional',{}))
                missing=needed-available
                assert not missing, (f'{class_type}の入力形式が変わっています: {sorted(missing)}。'
                    'エラーを記録してノートブックの更新が必要です。')
            required_inputs=set(defs['MiniMaxH3MotionContext']['input']['required'])
            missed=required_inputs-set(wf['106']['inputs'])
            assert not missed, f'MultiRefノードの必須入力が不足しています: {sorted(missed)}'
            print('✅ MultiRefノードの入力形式・必須項目を検証済み')
        (seg/'workflow_api.json').write_text(json.dumps(wf,ensure_ascii=False,indent=2))
        if idx==1: (JOB/'workflow_api.json').write_text(json.dumps(wf,ensure_ascii=False,indent=2))
        print(f'\n▶ H3 {idx}/{len(PLAN)}: {x["duration"]:.2f}秒 → {x["h3_frames"]} frames', flush=True)
        t0=time.perf_counter()
        response=requests.post(URL+'/prompt',json={'prompt':wf,'client_id':str(uuid.uuid4())},timeout=120)
        assert response.ok, f'H3 workflow失敗: {response.status_code} {response.text[:3000]}'
        info=response.json()
        assert 'prompt_id' in info, f'H3 queue拒否: {info}'
        pid=info['prompt_id']
        while True:
            time.sleep(10)
            hist=requests.get(URL+'/history/'+pid,timeout=60).json()
            if pid in hist:
                entry=hist[pid]
                status=entry.get('status',{})
                if status.get('status_str','success')!='success':
                    raise RuntimeError(f'H3 セグメント{idx}失敗: {status.get("messages",[])}')
                break
            print(f'  ⏳ セグメント{idx} 生成中 {((time.perf_counter()-t0)/60):.1f}分', flush=True)
        vids=[]
        for node in entry.get('outputs',{}).values():
            for v in node.get('videos',[]):
                name=v.get('filename')
                if name:
                    q=(COMFY/'output'/v.get('subfolder','')/name).resolve()
                    if q.is_file() and q.is_relative_to((COMFY/'output').resolve()): vids.append(q)
        if not vids:
            vids=sorted((COMFY/'output').rglob(f'H3_job_{JOB.name}_seg_{idx:03d}*.mp4'),
                key=lambda q:q.stat().st_mtime,reverse=True)
        assert vids, f'H3 セグメント{idx}の出力動画が見つかりません'
        shutil.copy2(vids[0], dst)
        measured=probe_duration(dst)
        if measured + 0.035 < x['duration']:
            raise RuntimeError(f'H3 セグメント{idx}の映像{measured:.3f}秒は音声{x["duration"]:.3f}秒より短いです。静止延長しません。')
        if H3_USE_CONTEXT:
            assert latent_file.is_file() and latent_file.stat().st_size>10000, f'H3 #{idx} のAV latent保存に失敗: {latent_file}'
        x['actual_h3_sec']=measured
        x['elapsed_h3_sec']=round(time.perf_counter()-t0,2)
        (JOB/'segments_plan.json').write_text(json.dumps(PLAN,ensure_ascii=False,indent=2))
        print(f'✅ H3 #{idx} 完了 {x["elapsed_h3_sec"]:.1f}秒 / 動画 {measured:.3f}秒', flush=True)
    H3_VIDEO=Path(PLAN[0]['h3_video'])
    RUN_TIMING={'status':'success','gpu':'A100 40GB','elapsed_sec':round(time.perf_counter()-h3_start,2),
        'segments':len(PLAN),'source_audio_sec':SOURCE_AUDIO_DURATION,'target_audio_sec':TARGET_AUDIO_SECONDS,
        'steps':NUM_STEPS,'motion_context':bool(H3_USE_CONTEXT),
        'continuity_mode':'context_loop_compatible' if H3_USE_CONTEXT else 'independent',
        'context_frames':H3_CONTEXT_FRAMES_INT if H3_USE_CONTEXT else 0,
        'context_reset_every':H3_CONTEXT_RESET_EVERY if H3_USE_CONTEXT else 0,
        'video_frames_per_segment':[x['h3_frames'] for x in PLAN]}
    (JOB/'generation_time.json').write_text(json.dumps(RUN_TIMING,ensure_ascii=False,indent=2))
    (JOB/'generation_time.txt').write_text('H3生成：'+json.dumps(RUN_TIMING,ensure_ascii=False,indent=2))
    print('\n✅ 全H3生成完了。⑦へ進んでください。')

In [ ]:
#@title ⑦【新規生成のみ】H3の映像をいったん確認
#@markdown **このセルの役割**：H3で作った映像（Context Loop互換長尺生成を含む）を元の日本語音声に合わせた比較用MP4にします。ここではまだ口元のLatentSync修正をしません。
#@markdown **ポイント**：完成動画ではありません。動き・服装・背景を確認し、問題なければ⑧に進みます。
#@markdown ---
if REUSE_H3:
    print('⏭ 既存H3動画モード：⑦-Bへ進んでください。')
else:
    import subprocess, json
    from pathlib import Path
    from IPython.display import display, FileLink, Video
    FINAL_ORIGINAL=JOB/'MiniMax_H3_with_original_audio.mp4'
    assert PLAN and all(Path(x['h3_video']).exists() for x in PLAN)
    preview_parts=[]
    for x in PLAN:
        sf=Path(x['segment_dir'])/'h3_preview_trim25.mp4'
        ff(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',x['h3_video'],
            '-vf',f'fps=25,trim=end_frame={x["frames_25"]},setpts=PTS-STARTPTS',
            '-an','-c:v','libx264','-r','25','-crf','18','-pix_fmt','yuv420p',str(sf)])
        preview_parts.append(sf)
    p_list=JOB/'h3_preview_concat.txt'
    p_list.write_text(''.join("file '"+str(x).replace("'","'\\''")+"'\n" for x in preview_parts))
    silent=JOB/'h3_preview_noaudio.mp4'
    ff(['ffmpeg','-y','-hide_banner','-loglevel','error','-f','concat','-safe','0','-i',str(p_list),
        '-c:v','copy','-an',str(silent)])
    ff(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(silent),'-i',str(AUDIO_PATH),
        '-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','192k',
        '-t',f'{TARGET_AUDIO_SECONDS:.6f}', '-movflags','+faststart',str(FINAL_ORIGINAL)])
    print(f'✅ H3プレビュー: {FINAL_ORIGINAL} / 合計{len(PLAN)}シーン')
    display(FileLink(str(FINAL_ORIGINAL)))
    if len(PLAN)==1 and FINAL_ORIGINAL.stat().st_size<30*1024**2:
        display(Video(str(FINAL_ORIGINAL),embed=True,width=560))
    print('👉 続いて⑧へ。H3の生成音声は最終MP4には使いません。')


In [ ]:
#@title ⑦-B【再開するとき】前のH3・手元のMP4を読み込む
#@markdown **このセルの役割**：②で『保存済みH3ジョブ』または『既存H3動画』を選んだときに使います。**H3の新規生成はしません。**
#@markdown **ポイント**：保存済みジョブは同じ元音声かを照合します。新規H3生成から進める場合、このセルは読み込みをしません。
#@markdown ---
import json, math, hashlib, wave, subprocess, datetime, shutil
from pathlib import Path
if REUSE_SAVED_JOB:
    outputs_root=WORK/'outputs'
    if SAVED_JOB_DIR.strip():
        JOB=Path(SAVED_JOB_DIR.strip()).expanduser()
    else:
        previous=globals().get('JOB')
        candidates=[]
        if previous: candidates.append(Path(previous))
        if outputs_root.exists():
            candidates+=sorted((p for p in outputs_root.iterdir() if p.is_dir()),
                               key=lambda p:p.stat().st_mtime_ns,reverse=True)
        JOB=next((p for p in candidates if (p/'source_mono_16k.wav').is_file()
                  and (p/'segments_plan.json').is_file()
                  and all(Path(z['h3_video']).is_file() for z in json.loads((p/'segments_plan.json').read_text()))),None)
        assert JOB, '保存済みH3ジョブが見つかりません。SAVED_JOB_DIRを指定してください。'
    assert (JOB/'source_mono_16k.wav').is_file(), '保存済み音声 source_mono_16k.wav がありません'
    PLAN=json.loads((JOB/'segments_plan.json').read_text())
    assert PLAN and all(Path(x['h3_video']).is_file() for x in PLAN), '保存済みH3動画が不足しています'
    assert all((Path(x['segment_dir'])/'mono_16k.wav').is_file() for x in PLAN),'保存済み分割音声がありません'
    MASTER=JOB/'source_mono_16k.wav'
    original_sec=sum(int(x['frames_25']) for x in PLAN)/25
    if abs(original_sec-TARGET_AUDIO_SECONDS)>0.041:
        raise RuntimeError(f'保存済みH3は{original_sec:.3f}秒分、今回の音声は{TARGET_AUDIO_SECONDS:.3f}秒です。音声尺を合わせてください。')
    # 音声の長さだけでなく内容も照合する。アップロードした音声が別の内容ならH3動画を誤って再利用しない。
    verify_audio=JOB/'verify_uploaded_audio_16k.wav'
    subprocess.run(['ffmpeg','-y','-hide_banner','-loglevel','error',
        '-i',str(AUDIO_PATH),'-t',f'{TARGET_AUDIO_SECONDS:.6f}', '-ac','1','-ar','16000','-c:a','pcm_s16le',str(verify_audio)],check=True)
    def pcm_sha(path):
        with wave.open(str(path),'rb') as wf:
            assert wf.getframerate()==16000 and wf.getnchannels()==1 and wf.getsampwidth()==2
            return hashlib.sha256(wf.readframes(wf.getnframes())).hexdigest()
    if pcm_sha(verify_audio)!=pcm_sha(MASTER):
        raise RuntimeError('保存済みH3で使用した音声と今回の音声が一致しません。間違った音声による再利用を防ぐため停止します。')
    verify_audio.unlink(missing_ok=True)
    H3_VIDEO=Path(PLAN[0]['h3_video'])
    SEG_DIR=JOB/'segments'
    RUN_TIMING=json.loads((JOB/'generation_time.json').read_text()) if (JOB/'generation_time.json').exists() else {'elapsed_sec':0}
    FINAL_ORIGINAL=JOB/'MiniMax_H3_with_original_audio.mp4'
    print(f'♻ 保存済みH3を再利用: {JOB} / {len(PLAN)}クリップ / 元音声PCM一致 ✅')
elif REUSE_EXTERNAL_VIDEO:
    H3_VIDEO=UPLOADED_H3_VIDEO
    src_sec=source_duration(H3_VIDEO)
    tgt=TARGET_AUDIO_SECONDS
    if src_sec+0.04<tgt and not ALLOW_FREEZE_WHEN_REUSING:
        raise RuntimeError(f'既存動画が{src_sec:.3f}秒、元音声が{tgt:.3f}秒。不足しているため停止します。')
    # 同じランタイムで再実行した際には外部動画ジョブも使い回す。
    # メディア名だけでなくSHA256も比較し、同名の別動画を誤再利用しない。
    def content_sha256(p):
        h=hashlib.sha256()
        with open(p,'rb') as f:
            for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
        return h.hexdigest()
    external_key={'video_sha256':content_sha256(H3_VIDEO),'audio_sha256':content_sha256(AUDIO_PATH),
       'target_seconds':round(tgt,6),'segment_seconds':MAX_SEGMENT_SECONDS,
       'allow_freeze':ALLOW_FREEZE_WHEN_REUSING}
    external_dir=WORK/'outputs';external_dir.mkdir(parents=True,exist_ok=True)
    options=[Path(globals().get('JOB'))] if globals().get('JOB') else []
    options+=sorted((q for q in external_dir.iterdir() if q.is_dir() and q.name.startswith('external_')),
                    key=lambda q:q.stat().st_mtime_ns,reverse=True)
    reused=next((q for q in options if (q/'external_job.json').is_file()
         and json.loads((q/'external_job.json').read_text())==external_key
         and (q/'segments_plan.json').is_file()),None)
    if reused:
        JOB=reused
        print('♻ 同一の外部MP4＋音声＋分割設定なので既存ジョブを再利用:',JOB)
    else:
        JOB=external_dir/('external_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        JOB.mkdir(parents=True,exist_ok=True)
        (JOB/'external_job.json').write_text(json.dumps(external_key,ensure_ascii=False,indent=2))
    SEG_DIR=JOB/'segments';SEG_DIR.mkdir(exist_ok=True)
    frame_total=math.ceil(tgt*25-1e-6)
    frame_max=math.floor(MAX_SEGMENT_SECONDS*25+1e-7)
    PLAN=[]
    for i,a in enumerate(range(0,frame_total,frame_max),1):
        b=min(frame_total,a+frame_max)
        sub=SEG_DIR/f'segment_{i:03d}';sub.mkdir(parents=True,exist_ok=True)
        PLAN.append({'index':i,'start_frame_25':a,'end_frame_25':b,'start_sec':a/25,
          'end_sec':b/25,'duration':(b-a)/25,'frames_25':b-a,
          'h3_frames':0,'h3_duration':0,'h3_video':str(H3_VIDEO),'segment_dir':str(sub)})
    (JOB/'segments_plan.json').write_text(json.dumps(PLAN,ensure_ascii=False,indent=2))
    RUN_TIMING={'status':'external_reuse','elapsed_sec':0,'segments':len(PLAN)}
    (JOB/'generation_time.json').write_text(json.dumps(RUN_TIMING,ensure_ascii=False,indent=2))
    FINAL_ORIGINAL=JOB/'MiniMax_H3_with_original_audio.mp4'
    print(f'♻ 外部MP4を再利用: {src_sec:.3f}秒、{len(PLAN)}分割。H3推論なし')
    if src_sec<tgt: print('⚠ 不足部分には末尾静止フレームを使います（明示的に許可された場合のみ）。')
else:
    print('✅ H3新規生成モード。⑥で生成した動画を引き継ぎます。')


In [ ]:
#@title ⑧【初回または再起動後】口パクAIの環境を準備
#@markdown **このセルの役割**：LatentSyncを動かすPython 3.10環境を用意します。H3で使ったComfyUIはここで終了させ、GPUメモリを空けます。
#@markdown **ポイント**：初回はライブラリの取得で時間がかかります。エラー時は元の内容とログを表示します。**同じランタイムなら基本的に一度でOK。**
#@markdown ---
import subprocess, time, os, sys
# Colab の MPLBACKEND=module://matplotlib_inline.backend_inline は
# 独立した Python 3.10 venv の matplotlib から見えない場合がある。
# 描画を必要としない LatentSync のサブプロセスでは Agg を使用する。
os.environ['MPLBACKEND'] = 'Agg'
print('✅ Matplotlibバックエンドを Agg に固定（LatentSync 用）', flush=True)
from pathlib import Path
import signal, shutil
# H3で占有したA100のVRAMを開放する。ComfyUIはセル実行停止だけでは終了しないため明示終了。
try:
    from psutil import process_iter
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','psutil'],check=True)
    from psutil import process_iter
stopped=[]
for proc in list(process_iter(['pid','cmdline'])):
    args=proc.info.get('cmdline') or []
    if any(str(a).endswith('/ComfyUI/main.py') for a in args):
        print('🛑 ComfyUIを停止:',proc.pid,flush=True)
        try:
            proc.terminate(); proc.wait(timeout=15)
        except Exception:
            try:proc.kill(); proc.wait(timeout=10)
            except Exception:pass
        stopped.append(proc.pid)
try: torch.cuda.empty_cache()
except Exception: pass
print('✅ ComfyUI停止数:',len(stopped))
print(subprocess.check_output(['nvidia-smi','--query-gpu=memory.used,memory.free','--format=csv,noheader'],text=True))
print('⚠ あとでH3を再生成したい場合は⑤でComfyUIを再起動してください。')
free_gb=shutil.disk_usage('/content').free/2**30
print(f'ディスク残量: {free_gb:.1f} GiB')
if free_gb < 17 and not (Path('/content/latentsync_py310/bin/python').is_file() and Path('/content/LatentSync/checkpoints/latentsync_unet.pt').is_file()):
    raise RuntimeError('LatentSyncの新規環境/モデル用に空きディスク17GiB以上を確保してください。')
if free_gb < 5:
    print('⚠ 残りディスク5GiB未満。複数設定の動画出力に注意してください。')
T0=time.perf_counter()
ROOT=Path('/content/LatentSync')
BASE=JOB/'latentsync'
BASE.mkdir(parents=True,exist_ok=True)
ENV=Path('/content/latentsync_py310')
PY=str(ENV/'bin/python')

def run(cmd,cwd=None,env=None):
    # 全ログを保存しつつ、失敗時に真のTracebackを表示する
    import collections
    cmd=[str(x) for x in cmd]
    print('▶', ' '.join(cmd),flush=True)
    log_path=BASE/'latentsync_setup.log'
    last=collections.deque(maxlen=65)
    with open(log_path,'a',encoding='utf-8') as log:
        log.write('\n>>> '+' '.join(cmd)+'\n')
        log.flush()
        p=subprocess.Popen(cmd,cwd=str(cwd) if cwd else None,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1)
        for line in p.stdout:
            log.write(line);log.flush()
            last.append(line.rstrip())
            # pip進捗も出すが極端なログは抑制
            if len(line)<700:print(line,end='',flush=True)
        rc=p.wait()
    if rc:
        print('\n'+'='*22+' 元のエラー出力（最後の65行） '+'='*22)
        print('\n'.join(last))
        print('ログ保存先:',log_path)
        raise RuntimeError(f'コマンド失敗（終了コード{rc}）。上のログに原因が表示されています: {cmd[0]}')

run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'])
run(['bash','-lc','command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y ffmpeg); command -v git >/dev/null || apt-get install -y git'])
if not ROOT.is_dir():
    run(['git','clone','--depth','1','https://github.com/bytedance/LatentSync.git',str(ROOT)])
if not Path(PY).is_file():
    run([sys.executable,'-m','pip','-q','install','uv'])
    run(['uv','python','install','3.10'])
    run(['uv','venv','--seed','--python','3.10',str(ENV)])

MARKER=ENV/'latentsync_deps_ready.txt'
if not MARKER.exists():
    # 前のノートブックでpipインストールだけ終わっている場合は、重い再インストールを避ける。
    already=subprocess.run([PY,'-m','pip','show','torch','torchvision','diffusers','transformers','insightface','onnxruntime-gpu'],
                           capture_output=True,text=True)
    names=['torch','torchvision','diffusers','transformers','insightface','onnxruntime-gpu']
    import re
    found_names=set(re.findall(r'^Name: (.+)$',already.stdout,re.MULTILINE))
    already_installed=(already.returncode==0 and all(name in found_names for name in names))
    if already_installed:
        print('✅ ライブラリはインストール済み。再ダウンロードせずに読み込みテストを実行します。')
    else:
        run([PY,'-m','pip','install','-U','pip','setuptools','wheel'])
        run([PY,'-m','pip','install','numpy==1.26.4','Cython==0.29.37','packaging'])
        # insightface builds from source: prepare numpy/cython first and install it separately.
        original=(ROOT/'requirements.txt').read_text()
        filtered='\n'.join(line for line in original.splitlines() if not line.lstrip().startswith('insightface=='))
        req=BASE/'requirements_colab.txt'
        req.parent.mkdir(parents=True,exist_ok=True)
        req.write_text(filtered+'\n')
        run([PY,'-m','pip','install','--no-cache-dir','-r',str(req)])
        run([PY,'-m','pip','install','--no-cache-dir','--no-build-isolation','insightface==0.7.3'])
        # GPU / CPU ONNXRuntime must not coexist; use the pinned GPU version.
        run([PY,'-m','pip','uninstall','-y','onnxruntime'])
        run([PY,'-m','pip','install','--no-cache-dir','--force-reinstall','--no-deps','onnxruntime-gpu==1.21.0'])
    # NumPy 2.x と scikit-image のC拡張のABI不整合を防ぐ。
    # insightface の依存解決がNumPyを2.xへ戻す場合があるため、すべてのpip処理の最後で固定する。
    # --no-depsを付け、固定済みのtorch・opencv・onnxruntimeなどには触れない。
    print('\n🩹 NumPy/scikit-imageバイナリ互換性を修復・固定します（Python 3.10環境のみ）', flush=True)
    run([PY,'-m','pip','install','--force-reinstall','--no-deps',
         'numpy==1.26.4','scikit-image==0.22.0'])
    # バージョンの修正後はライブラリを個別検査する。前のv3は複合importで止まり、
    # ユーザーが根本例外を見落としやすかったため、全てのエラーを最後まで出力する。
    print('\n🔍 ライブラリを個別に診断します。赤い「❌」の行と続くTracebackを確認してください。', flush=True)
    probe = 'import sys, importlib, importlib.metadata as md, traceback\nprint(\'====== LatentSync環境の詳細診断 ======\', flush=True)\nprint(\'Python:\', sys.executable, sys.version.split()[0], flush=True)\nfor package in [\'numpy\',\'scikit-image\',\'scipy\',\'opencv-python\',\'insightface\',\'onnxruntime-gpu\',\'torch\',\'torchvision\',\'transformers\',\'diffusers\']:\n    try: print(\'installed:\', package, md.version(package), flush=True)\n    except md.PackageNotFoundError: print(\'NOT INSTALLED:\', package, flush=True)\nerrors=[]\nchecks=[\n (\'NumPy\', \'import numpy; print(numpy.__version__)\'),\n (\'scikit-image geometry\', \'import skimage; import skimage._shared.geometry; print(skimage.__version__)\'),\n (\'scikit-image transform\', \'from skimage import transform; print("OK")\'),\n (\'scipy\', \'import scipy; print(scipy.__version__)\'),\n (\'OpenCV\', \'import cv2; print(cv2.__version__)\'),\n (\'InsightFace\', \'import insightface; from insightface.app import FaceAnalysis; print(insightface.__version__)\'),\n (\'ONNX Runtime\', \'import onnxruntime as ort; print(ort.__version__,ort.get_available_providers())\'),\n (\'PyTorch\', \'import torch; print(torch.__version__, torch.cuda.is_available())\'),\n (\'torchvision\', \'import torchvision; print(torchvision.__version__)\'),\n (\'transformers\', \'import transformers; print(transformers.__version__)\'),\n (\'diffusers\', \'import diffusers; print(diffusers.__version__)\'),\n]\nfor title, code in checks:\n    print(\'\\n== Checking:\',title,\'==\',flush=True)\n    try:\n        exec(code,{})\n        print(\'✅\', title, \'OK\', flush=True)\n    except BaseException as exc:\n        print(\'❌\',title,\'FAILED:\', type(exc).__name__,str(exc),flush=True)\n        traceback.print_exc(file=sys.stdout)\n        errors.append(title)\nprint(\'\\n=== 結果:\', \'失敗: \'+\', \'.join(errors) if errors else \'すべてのimportが成功\', \'===\',flush=True)\nsys.exit(1 if errors else 0)\n'
    try:
        run([PY, '-c', probe])
    except RuntimeError:
        print('\n🛑 LatentSyncのセットアップ検査が失敗しました。')
        print('⑥で生成したH3動画は削除されていません。')
        print('原因は上の「❌」以降のTracebackに表示されています。')
        print('全診断ログ:', BASE/'latentsync_setup.log')
        raise RuntimeError('LatentSync環境の診断で失敗しました。上の最初の「❌」とTracebackを参照してください。')
    MARKER.write_text(str(time.time()))
else:
    print('✅ 必要ライブラリのインストールは完了済み。')
print(f'環境準備の所要時間: {(time.perf_counter()-T0)/60:.1f} 分')

In [ ]:
#@title ⑨【初回のみ】口パクAIのモデルをダウンロード
#@markdown **このセルの役割**：LatentSync 1.6本体と、音声を解析するモデルを取得します。
#@markdown **ポイント**：ダウンロードしたファイルが残っていれば再取得を省略します。口パク設定を変えるだけなら再実行不要です。
#@markdown ---
from pathlib import Path
import subprocess,time
assert Path(PY).is_file(), '先に⑧を実行してください'
T0=time.perf_counter()
CKPT=ROOT/'checkpoints'
CKPT.mkdir(exist_ok=True)
models=[('latentsync_unet.pt',CKPT/'latentsync_unet.pt'),('whisper/tiny.pt',CKPT/'whisper/tiny.pt')]
for remote, local in models:
    if local.is_file() and local.stat().st_size>1_000_000:
        print('✅ キャッシュ使用:',remote, f'{local.stat().st_size/1024**3:.2f}GB')
        continue
    print('ダウンロード:',remote)
    run([PY,'-c',
         'from huggingface_hub import hf_hub_download; import sys; hf_hub_download(repo_id="ByteDance/LatentSync-1.6", filename=sys.argv[1], local_dir=sys.argv[2])',
         remote,str(CKPT)])
    assert local.is_file() and local.stat().st_size>1_000_000, f'モデルの取得に失敗: {local}'
# Diffusers will cache this VAE the first time; prefetch now to distinguish network errors.
run([PY,'-c',
     'from diffusers import AutoencoderKL; AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse",local_files_only=False); print("✅ SD VAE OK")'])
print(f'モデル準備の所要時間: {(time.perf_counter()-T0)/60:.1f} 分')

In [ ]:
#@title ⑩-A 動画と元音声を同じ長さにそろえる
#@markdown **このセルの役割**：LatentSyncに渡す前に動画を25fpsにそろえ、音声を各クリップに切り分けます。
#@markdown **ポイント**：動画・音声・分割設定が前回と同じなら前処理結果を再利用できます。**ここでは口パクをまだ生成しません。**
#@markdown ---
import json,math,subprocess,shutil,hashlib
from pathlib import Path
assert PLAN, '⑥または⑦-BでH3動画を用意してください'
assert 'BASE' in globals(), '⑧でLatentSync環境を準備してください'
SYNC_WORK=BASE/'work';SYNC_WORK.mkdir(parents=True,exist_ok=True)
REPORT=SYNC_WORK/'report.json'

def ffprobe(path):
    p=subprocess.run(['ffprobe','-v','error',
      '-show_entries','format=duration:stream=index,codec_type,nb_frames,duration,r_frame_rate',
      '-of','json',str(path)],capture_output=True,text=True,check=True)
    return json.loads(p.stdout)
def duration(path):
    return float(ffprobe(path)['format']['duration'])
def frame_count(path):
    raw=subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0',
      '-count_frames','-show_entries','stream=nb_read_frames',
      '-of','default=noprint_wrappers=1:nokey=1',str(path)],text=True).strip()
    return int(raw)
def fingerprint(path):
    p=Path(path);st=p.stat()
    return {'path':str(p.resolve()),'bytes':st.st_size,'mtime_ns':st.st_mtime_ns}

FFMPEG_LOG=JOB/'ffmpeg_v9.log'
def exec_logged(cmd):
    import sys
    cmd=[str(x) for x in cmd]
    with open(FFMPEG_LOG,'a',encoding='utf-8') as log:
        log.write('\n>>> '+' '.join(cmd)+'\n')
        result=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        if result.stdout:
            log.write(result.stdout)
            if result.returncode: print(result.stdout,flush=True)
        if result.returncode:
            print('FFmpegログ:',FFMPEG_LOG)
            raise subprocess.CalledProcessError(result.returncode,cmd,output=result.stdout)


SYNC_SEGMENTS=[]
for seg in PLAN:
    ix=int(seg['index']);sub=SYNC_WORK/f'clip_{ix:03d}';sub.mkdir(parents=True,exist_ok=True)
    vp=sub/'video_25fps.mp4';ap=sub/'audio_16k.wav'
    ref=Path(seg['h3_video']);wanted=int(seg['frames_25']);sec=wanted/25
    input_audio=Path(AUDIO_PATH) if REUSE_EXTERNAL_VIDEO else Path(seg['segment_dir'])/'mono_16k.wav'
    assert input_audio.exists(),f'音声クリップがありません: {input_audio}'
    key={'source_video':fingerprint(ref),'source_audio':fingerprint(input_audio),
         'frames_25':wanted,'start_frame_25':seg['start_frame_25'] if REUSE_EXTERNAL_VIDEO else 0,
         'external':REUSE_EXTERNAL_VIDEO,'allow_freeze':bool(ALLOW_FREEZE_WHEN_REUSING)}
    meta=sub/'aligned_source.json'
    cached=(meta.exists() and vp.is_file() and ap.is_file() and
      json.loads(meta.read_text())==key)
    if cached:
        try: cached=(frame_count(vp)==wanted and ap.stat().st_size>44)
        except Exception:cached=False
    if cached:
        print(f'♻ #{ix} 25fps動画・切出し音声を再利用')
    else:
        actual=duration(ref)
        end=seg['end_sec'] if REUSE_EXTERNAL_VIDEO else sec
        deficit=end-actual
        if deficit>0.045 and not (REUSE_EXTERNAL_VIDEO and ALLOW_FREEZE_WHEN_REUSING):
            raise RuntimeError(f'動画#{ix}が入力音声より{deficit:.3f}秒不足。H3を追加生成するか許可設定を変更してください。')
        vf='fps=25,'
        if REUSE_EXTERNAL_VIDEO and deficit>0.045:
            vf+=f'tpad=stop_mode=clone:stop_duration={deficit+0.12:.5f},'
        if REUSE_EXTERNAL_VIDEO:
            vf+=f'trim=start_frame={seg["start_frame_25"]}:end_frame={seg["end_frame_25"]},setpts=PTS-STARTPTS'
        else:
            vf+=f'trim=end_frame={wanted},setpts=PTS-STARTPTS'
        exec_logged(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(ref),'-vf',vf,
          '-an','-c:v','libx264','-r','25','-crf','18','-pix_fmt','yuv420p',str(vp)])
        got=frame_count(vp)
        assert got==wanted, f'映像の25fpsフレーム不足: segment {ix} got={got}, wanted={wanted}'
        if REUSE_EXTERNAL_VIDEO:
            # カットを元音声から直接実行。端数は無音補完してフレーム境界に一致させる。
            start=seg['start_frame_25']/25
            exec_logged(['ffmpeg','-y','-hide_banner','-loglevel','error',
              '-i',str(AUDIO_PATH),'-af',f'atrim=start={start:.6f}:duration={sec:.6f},asetpts=PTS-STARTPTS,aresample=16000,apad=whole_dur={sec:.6f},atrim=duration={sec:.6f}',
              '-ac','1','-ar','16000','-c:a','pcm_s16le',str(ap)])
        else:shutil.copy2(input_audio,ap)
        meta.write_text(json.dumps(key,ensure_ascii=False,indent=2))
        print(f'✅ #{ix} 25fps映像 {got}フレーム / 音声 {sec:.3f}秒')
    SYNC_SEGMENTS.append({'index':ix,'video':str(vp),'audio':str(ap),
        'frames_25':wanted,'duration':sec})
INPUT_VIDEO=Path(SYNC_SEGMENTS[0]['video']);INPUT_AUDIO=Path(SYNC_SEGMENTS[0]['audio'])
target_sec=TARGET_AUDIO_SECONDS
report={'pipeline':'MiniMax H3 Ref2VA -> LatentSync 1.6','mode':PIPELINE_MODE,
  'source_audio_seconds':SOURCE_AUDIO_DURATION,'target_audio_seconds':target_sec,
  'video_frames_25_total':sum(x['frames_25'] for x in SYNC_SEGMENTS),
  'target_video_seconds_25fps':sum(x['frames_25'] for x in SYNC_SEGMENTS)/25,
  'segment_count':len(SYNC_SEGMENTS),'configurations':VARIANTS}
REPORT.write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(f'📏 元音声 {target_sec:.3f}秒、25fps映像 {report["target_video_seconds_25fps"]:.3f}秒（最終誤差40ms以内）')


In [ ]:
#@title ⑩-B 口パク処理の前に顔を見つけられるか検査
#@markdown **このセルの役割**：本格的なGPU推論の前に、動画の複数フレームで顔の位置を検出できるか確認します。
#@markdown **ポイント**：顔が小さい・横向き・隠れている場合は失敗することがあります。成功したら⑪へ進みます。
#@markdown ---
import os,textwrap,subprocess
from pathlib import Path
assert Path(INPUT_VIDEO).is_file() and Path(PY).is_file()
print(f'🧪 {len(SYNC_SEGMENTS)}クリップの顔検出テストを行います。')
FACE_CHECK=SYNC_WORK/'check_face.py'
FACE_CHECK.write_text(textwrap.dedent("""
import os, sys, cv2, torch, onnxruntime as ort
from latentsync.utils.face_detector import FaceDetector
print('CUDA available:',torch.cuda.is_available(),flush=True)
print('ONNX runtime providers:',ort.get_available_providers(),flush=True)
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('onnxruntime-gpuのCUDAExecutionProviderが使えません。CUDAライブラリを確認してください。')
detector=FaceDetector(device='cuda')
all_results=[]
for path in sys.argv[1:]:
    print('CHECK',path,flush=True)
    cap=cv2.VideoCapture(path);count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if count<1: raise RuntimeError('動画を読めません: '+path)
    indices=[min(max(0,int(count*f)),count-1) for f in (0.15,0.5,0.85)]
    for ix in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES,ix); ok,frame=cap.read()
        if not ok: raise RuntimeError(f'フレーム読込失敗: {path} frame {ix}')
        frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        box,landmarks=detector(frame)
        all_results.append(box is not None)
        print(f'frame {ix}:', '✅ 顔認識OK' if box is not None else '❌ 顔認識なし',flush=True)
    cap.release()
if not all(all_results):
    raise RuntimeError('顔検出失敗。顔の小ささ、横顔、アニメ調などが原因の可能性があります。')
print('✅ 全フレームで顔検出成功。推論を実行できます。')
""").lstrip())
# Note: launch as a separate Python process to use pinned Python 3.10 dependencies.
env=os.environ.copy()
site=list((ENV/'lib').glob('python3.10/site-packages'))
if site:
    lib_paths=list(site[0].glob('nvidia/*/lib'))
    env['LD_LIBRARY_PATH']=':'.join(map(str,lib_paths))+':'+env.get('LD_LIBRARY_PATH','')
env['PYTHONPATH']=str(ROOT)+os.pathsep+env.get('PYTHONPATH','')
try:
    run([PY,str(FACE_CHECK)]+[x['video'] for x in SYNC_SEGMENTS],cwd=ROOT,env=env)
except subprocess.CalledProcessError:
    print('顔検出に失敗した場合：人物の顔が大きめ・正面に映った動画を使ってください。')
    raise


In [ ]:
#@title ⑪【GPU使用】日本語音声に合わせて口を動かす
#@markdown **このセルの役割**：LatentSyncが元音声に合わせて口元を修正します。比較ONなら、②で選んだ設定数だけMP4用素材を作ります。
#@markdown **ポイント**：**H3の動画は再生成しません。** 同じ入力・設定の結果はキャッシュを再利用します。口パク調整で主に時間がかかる工程です。
#@markdown ---
import subprocess, os, time, json, hashlib, collections
from pathlib import Path
assert SYNC_SEGMENTS, '⑩-Aを先に実行してください'
assert (ROOT/'checkpoints/latentsync_unet.pt').is_file(), '⑨でモデルを用意してください'
free_raw=subprocess.check_output(['nvidia-smi','--query-gpu=memory.free','--format=csv,noheader,nounits'],text=True)
free_mib=max(int(s.strip()) for s in free_raw.splitlines() if s.strip())
print(f'GPU空きVRAM: {free_mib/1024:.1f}GiB')
if free_mib<18000:
    raise RuntimeError(f'LatentSync 1.6の空きVRAMが不足しています（{free_mib} MiB）。ComfyUIを停止してください。')
# ⑩-Bを実行しない場合も動くようCUDAライブラリパスをここで構成する。
env=os.environ.copy();env['MPLBACKEND']='Agg'
site=list((ENV/'lib').glob('python3.10/site-packages'))
if site:
    libs=list(site[0].glob('nvidia/*/lib'))
    env['LD_LIBRARY_PATH']=':'.join(map(str,libs))+':'+env.get('LD_LIBRARY_PATH','')
env['PYTHONPATH']=str(ROOT)+os.pathsep+env.get('PYTHONPATH','')

MODEL_ID=fingerprint(ROOT/'checkpoints/latentsync_unet.pt')
VARIANT_OUTPUTS=[]
for variant in VARIANTS:
    tag=variant['tag'];destdir=SYNC_WORK/'variants'/tag;destdir.mkdir(parents=True,exist_ok=True)
    video_parts=[];used_cache=0;sum_elapsed=0.0
    for clip in SYNC_SEGMENTS:
        ix=clip['index'];raw=destdir/f'clip_{ix:03d}_raw.mp4';metadata=destdir/f'clip_{ix:03d}_meta.json'
        signature={'input_video':fingerprint(clip['video']),'input_audio':fingerprint(clip['audio']),
          'model':MODEL_ID,'steps':variant['steps'],'guidance':variant['guidance'],
          'deepcache':variant['deepcache'],'seed':SEED+ix-1}
        cached=(raw.is_file() and raw.stat().st_size>10000 and metadata.exists()
                and json.loads(metadata.read_text()).get('signature')==signature)
        if cached:
            print(f'♻ {tag} #{ix}/{len(SYNC_SEGMENTS)} キャッシュ再利用')
            used_cache+=1;elapsed=0.0
        else:
            cmd=[PY,'-m','scripts.inference',
              '--unet_config_path','configs/unet/stage2_512.yaml',
              '--inference_ckpt_path','checkpoints/latentsync_unet.pt',
              '--inference_steps',str(variant['steps']),
              '--guidance_scale',str(variant['guidance']),
              '--seed',str(SEED+ix-1),'--video_path',clip['video'],
              '--audio_path',clip['audio'],'--video_out_path',str(raw),
              '--temp_dir',str(destdir/f'temp_{ix:03d}')]
            if variant['deepcache']:cmd.append('--enable_deepcache')
            print(f'▶ LatentSync {tag} #{ix}/{len(SYNC_SEGMENTS)} {clip["duration"]:.2f}秒',flush=True)
            start=time.perf_counter()
            # setupセルのrunはログ保存・元Traceback表示に対応
            run(cmd,cwd=ROOT,env=env)
            elapsed=time.perf_counter()-start
            assert raw.is_file() and raw.stat().st_size>10000,f'{tag}: セグメント{ix}の出力がありません'
            metadata.write_text(json.dumps({'signature':signature,'elapsed_sec':round(elapsed,2)},ensure_ascii=False,indent=2))
            print(f'✅ {tag} #{ix} {elapsed:.1f}秒')
        sum_elapsed+=elapsed
        video_parts.append({'index':ix,'video':str(raw),'frames_25':clip['frames_25'],'duration':clip['duration']})
    VARIANT_OUTPUTS.append({'variant':variant,'tag':tag,'parts':video_parts,
                            'elapsed_sec':round(sum_elapsed,2),'reused_clips':used_cache})
report['variants']=[{'tag':v['tag'],'steps':v['variant']['steps'],
    'guidance':v['variant']['guidance'],'deepcache':v['variant']['deepcache'],
    'inference_seconds_this_run':v['elapsed_sec'],'cache_hits':v['reused_clips']}
    for v in VARIANT_OUTPUTS]
REPORT.write_text(json.dumps(report,ensure_ascii=False,indent=2))
print('✅',len(VARIANT_OUTPUTS),'設定のLatentSync処理完了。⑫で結合して比較します。')


In [ ]:
#@title ⑫ 完成動画を確認・保存する｜MP4・比較動画・ZIP
#@markdown **このセルの役割**：分割クリップをつなぎ、元の日本語音声を付けた完成MP4とレポートを作成します。比較ONなら横並び動画も出力します。
#@markdown **ポイント**：画面に出るファイルリンクから個別に保存できます。ZIP一括取得も可能です。**動画の長さが合っていても口パクの自然さは目視で確認**してください。
#@markdown ---
import json,zipfile,subprocess,time,os,math
from pathlib import Path
from IPython.display import display,Video,FileLink,HTML
from google.colab import files
from html import escape
assert VARIANT_OUTPUTS, '⑪を先に実行してください'
def ffmpeg_logged(cmd):
    cmd=[str(x) for x in cmd]
    logfile=JOB/'ffmpeg_v9.log'
    with open(logfile,'a',encoding='utf-8') as log:
        log.write('\n>>> '+' '.join(cmd)+'\n')
        result=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        if result.stdout:log.write(result.stdout)
    if result.returncode:
        print(result.stdout,flush=True)
        print('FFmpegログ:',logfile)
        raise subprocess.CalledProcessError(result.returncode,cmd,output=result.stdout)

TOTAL_FRAMES=sum(x['frames_25'] for x in SYNC_SEGMENTS)
FINAL_FILES=[]
for item in VARIANT_OUTPUTS:
    tag=item['tag'];variant=item['variant']
    out=JOB/f'H3_LatentSync_{tag}_Final.mp4'
    normalized=[]
    for part in item['parts']:
        src=Path(part['video']);dst=src.with_name(src.stem+'_exact25.mp4')
        need=part['frames_25']
        # LatentSyncの生成fpsに関係なく、25fpsへ変換した後で必要フレーム数を検証。
        cmd=['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(src),
          '-vf',f'fps=25,trim=end_frame={need},setpts=PTS-STARTPTS',
          '-an','-c:v','libx264','-r','25','-crf','18','-pix_fmt','yuv420p',str(dst)]
        if not dst.is_file() or dst.stat().st_mtime_ns<src.stat().st_mtime_ns:
            ffmpeg_logged(cmd)
        got=frame_count(dst)
        assert got==need, f'動画が不足: {tag} segment{part["index"]} got={got}/{need}'
        normalized.append(dst)
    lst=JOB/f'concat_{tag}.txt'
    lst.write_text(''.join("file '"+str(p).replace("'","'\\''")+"'\n" for p in normalized))
    silent=JOB/f'joined_{tag}_silent.mp4'
    ffmpeg_logged(['ffmpeg','-y','-hide_banner','-loglevel','error','-f','concat','-safe','0',
      '-i',str(lst),'-an','-c:v','copy',str(silent)])
    assert frame_count(silent)==TOTAL_FRAMES, f'{tag}連結後のフレーム数が不一致'
    ffmpeg_logged(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(silent),'-i',str(AUDIO_PATH),
      '-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','192k',
      '-af',f'atrim=duration={target_sec:.6f},asetpts=PTS-STARTPTS',
      '-movflags','+faststart',str(out)])
    assert out.stat().st_size>10000
    secs=duration(out)
    item['final_mp4']=str(out);item['final_seconds']=secs
    FINAL_FILES.append(out)
    print(f'✅ {tag}: {secs:.3f}秒（元音声{target_sec:.3f}秒、差{secs-target_sec:+.3f}秒）')

COMPARE_MP4=None
if CREATE_SIDE_BY_SIDE and len(FINAL_FILES)>1:
    chosen=FINAL_FILES[:3]
    # 比較ファイルは低解像度にしてGPU CUとファイルサイズを抑える（CPUのFFmpegだけで処理）。
    filt=';'.join(f'[{i}:v]scale=480:-2,setsar=1[v{i}]' for i in range(len(chosen)))
    filt+=';'+''.join(f'[v{i}]' for i in range(len(chosen)))+f'hstack=inputs={len(chosen)}[v]'
    COMPARE_MP4=JOB/'comparison_side_by_side.mp4'
    cmd=['ffmpeg','-y','-hide_banner','-loglevel','error']
    for p in chosen:cmd+=['-i',str(p)]
    cmd+=['-filter_complex',filt,'-map','[v]','-map','0:a:0',
         '-c:v','libx264','-preset','veryfast','-crf','24','-pix_fmt','yuv420p',
         '-c:a','copy','-shortest',str(COMPARE_MP4)]
    ffmpeg_logged(cmd)
    (JOB/'comparison_order.txt').write_text('左から順に: '+', '.join(x['tag'] for x in VARIANT_OUTPUTS[:3])+'\n')
    print('✅ 横並び比較:',COMPARE_MP4.name,'（左から',', '.join(x['tag'] for x in VARIANT_OUTPUTS[:3]),'）')

report['results']=[{'tag':x['tag'],'steps':x['variant']['steps'],'guidance_scale':x['variant']['guidance'],
  'deepcache':x['variant']['deepcache'],'model_inference_seconds_this_run':x['elapsed_sec'],
  'cache_hits':x['reused_clips'],'final_mp4':Path(x['final_mp4']).name,
  'final_seconds':x['final_seconds'],'drift_vs_audio_sec':round(x['final_seconds']-target_sec,4)}
  for x in VARIANT_OUTPUTS]
report['h3_generation_seconds_previous_run']=RUN_TIMING.get('elapsed_sec',0)
report['model_inference_seconds_this_run_total']=round(sum(x['elapsed_sec'] for x in VARIANT_OUTPUTS),2)
report['comparison_video']=COMPARE_MP4.name if COMPARE_MP4 else None
REPORT.write_text(json.dumps(report,ensure_ascii=False,indent=2))
SUMMARY=JOB/'comparison_report.json'
SUMMARY.write_text(json.dumps(report,ensure_ascii=False,indent=2))
ZIP=JOB/'H3_LatentSync_Results.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    paths=list(FINAL_FILES)+[REPORT,JOB/'generation_time.json',JOB/'segments_plan.json']
    if Path(FINAL_ORIGINAL).is_file():paths.append(Path(FINAL_ORIGINAL))
    if COMPARE_MP4:paths += [COMPARE_MP4,JOB/'comparison_order.txt']
    added=set()
    for p in paths:
        if p.is_file() and p.resolve() not in added:
            z.write(p,p.name);added.add(p.resolve())
    if INCLUDE_RAW_H3_IN_ZIP:
        for seg in PLAN:
            p=Path(seg['h3_video'])
            if p.is_file() and p.resolve() not in added:
                z.write(p,f'raw_h3/segment_{seg["index"]:03d}.mp4')
                added.add(p.resolve())
print(f'📦 ZIP {ZIP.stat().st_size/1024**2:.1f} MiB:',ZIP)
rows=[('設定','steps','guidance','DeepCache','今実行した推論(秒)','キャッシュ数','完成尺(秒)','対音声差(秒)')]
for item in VARIANT_OUTPUTS:
    v=item['variant']
    rows.append((item['tag'],str(v['steps']),f'{v["guidance"]:.2f}',str(v['deepcache']),
      f'{item["elapsed_sec"]:.1f}',str(item['reused_clips']),
      f'{item["final_seconds"]:.3f}',f'{item["final_seconds"]-target_sec:+.3f}'))
table='<h3>🎬 LatentSync設定比較（元音声 %.3f秒）</h3>'%target_sec
for n,row in enumerate(rows):
    cell='th' if n==0 else 'td'
    table+='<tr>'+''.join(f'<{cell} style="border:1px solid #bbb;padding:7px">{escape(str(x))}</{cell}>' for x in row)+'</tr>'
display(HTML('<table style="border-collapse:collapse">'+table+'</table>'))
for f in FINAL_FILES:display(FileLink(str(f)))
if COMPARE_MP4:display(FileLink(str(COMPARE_MP4)))
display(FileLink(str(REPORT)));display(FileLink(str(ZIP)))
if len(FINAL_FILES)==1 and FINAL_FILES[0].stat().st_size<25*1024**2:
    display(Video(str(FINAL_FILES[0]),embed=True,width=560))
if AUTO_DOWNLOAD_ZIP: files.download(str(ZIP))


## ✅ 完成したら確認すること

最後の **⑫** で生成された `H3_LatentSync_..._Final.mp4` が完成動画です。まず音を出して再生し、**①元音声の内容、②口の動き、③顔の安定性、④手と背景の崩れ、⑤動画の最後が切れていないか**を確認してください。

### おすすめの使い方

- **初回の動作確認**：②で `TEST_SECONDS=15`、`COMPARE_LIPSYNC=False`。完成版をまず1本作り、正しく再生できるかを見る。
- **口パクが気になる場合**：H3再開モードにして `COMPARE_LIPSYNC=True`、`COMPARE_STEPS='20'`、`COMPARE_GUIDANCES='1.5,1.8,2.0'`。既存のH3を使って3種類を比較する。
- **本番の全尺生成**：気に入った設定で比較モードをOFFに戻し、`TEST_SECONDS=0`。v13はMotion Contextでクリップ間の動きを継続。まず30～45秒で継ぎ目を確認し、問題なければ全尺へ。

### よくある質問

**Q. 同じH3映像で口の形だけ変更したい。**

A. ②で「保存済みH3ジョブから再開」または「既存H3動画から再開」を選び、該当する入力を①で用意します。③～⑦のH3生成を省けます。すでに⑧・⑨が済んでいる同じランタイムなら再インストールも通常不要です。

**Q. `LIPSYNC_GUIDANCE`を変更したのに効果がない。**

A. 比較モード（`COMPARE_LIPSYNC`）がONの場合、通常用の `LIPSYNC_GUIDANCE` は使いません。代わりに `COMPARE_GUIDANCES` を変更してください。

**Q. 生成結果をまた使いたい。**

A. Colabの `/content` は永続保存ではありません。必要な完成MP4・比較MP4・ZIPは⑫からPCに保存してください。H3の生動画を後日再利用したい場合は、H3動画のMP4自体も保存しておくと便利です。

**Q. エラーが出た。**

A. エラーの冒頭だけではなく、**元のTraceback全体**と実行セル番号を控えてください。⑧～⑪は `latentsync_setup.log`、動画処理は `ffmpeg_v9.log` にログが残る場合があります。動いているところまでやり直さず、まず失敗したセルを確認してください。

---

**v13の改訂範囲：Motion Context拡張・latent保存/読込・重複トリム・音声尺に合わせた自動分割。** 15秒以下およびLatentSync推論部はv12の方式を維持しています。A100での連続生成の実動作は未検証です。

### 長尺モードの注意

- **生成途中でColabランタイムが切れると、`/content` 内の潜在データが消える場合があります。** 動画の続き生成には直前の潜在データ（`.safetensors`）も必要です。
- 保存済みの動画MP4だけをアップロードした場合、②の再開モードで**LatentSyncによる口パク修正はできます**が、Motion Contextの潜在データの連続性は復元されません。
- 内部のH3音声が連続していても、完成動画に採用するのは常に入力した元音声です。
- 継ぎ目や音声との同期、人物の一貫性は動画を必ず見て確認してください。


- 長尺で顔が少しずつ変わる場合は、②の **「何クリップごとに元画像へ戻すか」** を **2** または **3** にすると改善しやすいです。

**v14.1から**：セグメント2で `Required input is missing: context_frames / encode_mode / ...` と表示される問題を修正しています。Colabの同じランタイムで再実行する場合、完了済みのセグメント1を再利用できます。